In [1]:
pip install requests beautifulsoup4

   ━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━ 3/8 [idna]  WARNING: The script normalizer is installed in '/Library/Frameworks/Python.framework/Versions/3.14/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [beautifulsoup4]m [beautifulsoup4]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import unquote

def download_pdfs_from_html(html_content):
    # 1. Setup download directory
    download_dir = "ndc_downloads_robust"
    if not os.path.exists(download_dir):
        os.makedirs(download_dir)
        print(f"Directory created: {download_dir}")

    soup = BeautifulSoup(html_content, "html.parser")
    
    # 2. Extract Links
    pdf_links = set()
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        if href.lower().strip().endswith('.pdf'):
            pdf_links.add(href.strip())

    print(f"Found {len(pdf_links)} unique PDF links.")

    # 3. Define Headers to mimic a real browser (Chrome on Windows)
    # The 'Referer' header is often critical for government/UN sites to prove 
    # you came from their own page and are not just a bot hotlinking the file.
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        "Referer": "https://unfccc.int/NDCREG",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    }

    # 4. Download loop
    for url in pdf_links:
        try:
            filename = unquote(url.split('/')[-1])
            file_path = os.path.join(download_dir, filename)

            print(f"Attempting to download: {filename}...")

            # Use a session to persist cookies if necessary
            with requests.Session() as s:
                response = s.get(url, headers=headers, stream=True, timeout=15)
                response.raise_for_status()

                # --- VALIDATION CHECK ---
                # We read the first 4 bytes. A real PDF always starts with %PDF
                # We peek at the content without consuming it entirely yet.
                file_start = next(response.iter_content(chunk_size=4))
                
                # Check if it looks like a PDF
                if file_start.startswith(b'%PDF'):
                    with open(file_path, 'wb') as f:
                        f.write(file_start) # Write the first 4 bytes we peeked
                        # Write the rest of the stream
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"  [SUCCESS] Saved to {file_path}")
                else:
                    # If it's not %PDF, it's likely an HTML error page (Access Denied)
                    print(f"  [BLOCKED] The server returned an HTML page instead of a PDF.")
                    print(f"            This usually means the UNFCCC firewall blocked the script.")
                    print(f"            First few bytes received: {file_start}")

            # Sleep for 1 second to be polite and avoid rate-limiting
            time.sleep(1)

        except requests.exceptions.RequestException as e:
            print(f"  [ERROR] Connection failed: {e}")
        except Exception as e:
            print(f"  [ERROR] Unexpected error: {e}")

    print("\nProcessing complete.")


# The HTML source provided in your prompt
html_source = """
<div class="view-content">
<div class="table-responsive">
<table class="table table-hover table-striped">
<thead>
<tr class="processed">
<th id="view-title-table-column" class="views-field views-field-title" scope="col"><a href="?field_party_region_target_id=All&amp;field_document_ca_target_id=All&amp;field_vd_status_target_id=5933&amp;start_date_datepicker=&amp;end_date_datepicker=&amp;order=title&amp;sort=asc" title="sort by Party">Party</a></th>
<th id="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments" scope="col">Title</th>
<th id="view-nothing-table-column" class="views-field views-field-nothing" scope="col">Language</th>
<th id="view-nothing-1-table-column" class="views-field views-field-nothing-1" scope="col">Translation</th>
<th id="view-field-version-number-table-column" class="views-field views-field-field-version-number" scope="col"><a href="?field_party_region_target_id=All&amp;field_document_ca_target_id=All&amp;field_vd_status_target_id=5933&amp;start_date_datepicker=&amp;end_date_datepicker=&amp;order=field_version_number&amp;sort=desc" title="sort by Version">Version</a></th>
<th id="view-field-vd-status-table-column" class="views-field views-field-field-vd-status" scope="col">Status</th>
<th id="view-field-document-sb-table-column" aria-sort="descending" class="views-field views-field-field-document-sb is-active" scope="col"><a href="?field_party_region_target_id=All&amp;field_document_ca_target_id=All&amp;field_vd_status_target_id=5933&amp;start_date_datepicker=&amp;end_date_datepicker=&amp;order=field_document_sb&amp;sort=asc" title="sort by Submission Date">Submission Date<span class="icon glyphicon glyphicon-chevron-down icon-after" aria-hidden="true" data-toggle="tooltip" data-placement="bottom" title="" data-original-title="Sort ascending"></span>
</a></th>
<th id="view-nothing-2-table-column" class="views-field views-field-nothing-2" scope="col">Additional documents</th>
</tr>
</thead>
<tbody>
<tr class="submission-nid-655674 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cog_flag_0.gif?h=27ba19da&amp;itok=krp2rQDV" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Congo
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2026-02/CDN%203.0%20de%20la%20R%C3%A9publique%20du%20Congo%20version%20finale.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-02/CDN%203.0%20de%20la%20R%C3%A9publique%20du%20Congo%20version%20finale.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-02/M.%20Simon%20Stiell%20_Secre%CC%81taire%20Exe%CC%81cutif%20de%20la%20Convention%20Cadre%20des%20Nations%20Unies%20sur%20le%20Changement%20Climatique%20_Ts%20de%20la%20CDN%203.0%20en%20Re%CC%81publique%20du%20Congo.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Submission letter</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/02/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-02/CDN%203.0%20de%20la%20R%C3%A9publique%20du%20Congo%20version%20finale.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-02/M.%20Simon%20Stiell%20_Secre%CC%81taire%20Exe%CC%81cutif%20de%20la%20Convention%20Cadre%20des%20Nations%20Unies%20sur%20le%20Changement%20Climatique%20_Ts%20de%20la%20CDN%203.0%20en%20Re%CC%81publique%20du%20Congo.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Submission letter</a></div>
</td>
</tr>
<tr class="submission-nid-655523 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/plw_flag.gif?h=fe1b3b8f&amp;itok=fiPF38b3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Palau
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Palau%20Final%20Endorsed%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Palau NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Palau%20Final%20Endorsed%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Palau NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Palau%20Final%20Endorsed%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Palau NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655496 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tto_flag.gif?h=27ba19da&amp;itok=NAuBcAoy" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Trinidad and Tobago
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Trinidad%20and%20Tobago%20Second%20NDC%20%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago Second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Trinidad%20and%20Tobago%20Second%20NDC%20%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-01/Letter%20to%20the%20UNFCCC%20-%20Submission%20of%20T%26T%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Letter of submission Trinidad and Tobago</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Trinidad%20and%20Tobago%20Second%20NDC%20%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-01/Letter%20to%20the%20UNFCCC%20-%20Submission%20of%20T%26T%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Letter of submission Trinidad and Tobago</a></div>
</td>
</tr>
<tr class="submission-nid-655445 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hnd_flag.gif?h=d4b38aaf&amp;itok=R8mDwun-" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Honduras
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2026-01/HON%20NDC%203.0%202026%20Oficial.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras NDC 3.0 </a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/HON%20NDC%203.0%202026%20Oficial.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">21/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/HON%20NDC%203.0%202026%20Oficial.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655394 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/wsm_flag.gif?h=b650b131&amp;itok=5PKln8Kt" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Samoa
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Samoa%20NDC3.0_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Samoa%20NDC3.0_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">14/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Samoa%20NDC3.0_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655361 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nru_flag.gif?h=da9490b2&amp;itok=VsITSzig" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Nauru
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Nauru%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Nauru%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Nauru%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655391 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/arm_flag.gif?h=57cf074e&amp;itok=3Ug20-D3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Armenia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Armenia%27s%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Armenia%27s%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Armenia%27s%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655360 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bfa_flag.gif?h=57cf074e&amp;itok=u4hpEJnZ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Burkina Faso
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2026-01/BKF-CDN%203.0_BURKINA%20FASO.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/BKF-CDN%203.0_BURKINA%20FASO.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/BKF-CDN%203.0_BURKINA%20FASO.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655320 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sau_flag.gif?h=57cf074e&amp;itok=ZaKxwcmX" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Saudi Arabia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/2nd%20KSA%20NDC%20Document%20Final%20Document%20%2828122025%29_DNA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia Second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/2nd%20KSA%20NDC%20Document%20Final%20Document%20%2828122025%29_DNA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/2nd%20KSA%20NDC%20Document%20Final%20Document%20%2828122025%29_DNA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655301 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kor_flag.gif?h=57cf074e&amp;itok=TMP5hws2" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Republic of Korea
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-12/The%20Republic%20of%20Koreas%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of Korea's 2035 NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/The%20Republic%20of%20Koreas%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of Korea's 2035 NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/The%20Republic%20of%20Koreas%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of Korea's 2035 NDC</a></div>
</td>
</tr>
<tr class="submission-nid-655299 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/slv_flag.gif?h=f3674879&amp;itok=GZlrX9HT" width="57" height="35" alt="" typeof="Image" class="img-responsive">
El Salvador
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-12/NDC%20EL%20SALVADOR%202025-%20VF.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC%20EL%20SALVADOR%202025-%20VF.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC%20EL%20SALVADOR%202025-%20VF.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655298 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gab_flag.gif?h=d4b38aaf&amp;itok=sL8Y0n4I" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Gabon
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-12/NDC3.0_Gabon_5_11_2025-final%20version.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Gabon NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC3.0_Gabon_5_11_2025-final%20version.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Gabon NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC3.0_Gabon_5_11_2025-final%20version.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Gabon NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655274 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/col_flag.gif?h=57cf074e&amp;itok=eqy94Nxd" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Colombia
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-12/NDC%203.0%20Transformaciones%20para%20la%20Vida%20%28Colombia%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC%203.0%20Transformaciones%20para%20la%20Vida%20%28Colombia%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC%203.0%20Transformaciones%20para%20la%20Vida%20%28Colombia%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655679 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/zmb_flag.gif?h=57cf074e&amp;itok=Uq9A4N5D" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Zambia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-02/1Final%20Submission%20of%20Zambia%20NDC%203.0%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-02/1Final%20Submission%20of%20Zambia%20NDC%203.0%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">15/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-02/1Final%20Submission%20of%20Zambia%20NDC%203.0%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655182 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sle_flag.gif?h=57cf074e&amp;itok=Fmqmw1rH" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sierra Leone
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-12/Sierra%20Leone%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/Sierra%20Leone%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/Sierra%20Leone%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655178 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/rwa_flag.gif?h=57cf074e&amp;itok=aCAcWVpz" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Rwanda
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-12/Rwanda%20NDC3.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/Rwanda%20NDC3.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/Rwanda%20NDC3.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655117 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bhr_flag.gif?h=d4b38aaf&amp;itok=cJULjQL0" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bahrain
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-12/30112025_Bahrain_2025NDC3.0_vSubmitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/30112025_Bahrain_2025NDC3.0_vSubmitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">01/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/30112025_Bahrain_2025NDC3.0_vSubmitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655104 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pan_flag.gif?h=57cf074e&amp;itok=QSPIis5C" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Panama
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/Pacto%20de%20Panam%C3%A1%20con%20la%20Naturaleza%20%28Nature%20Pledge%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Pacto%20de%20Panam%C3%A1%20con%20la%20Naturaleza%20%28Nature%20Pledge%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Pacto%20de%20Panam%C3%A1%20con%20la%20Naturaleza%20%28Nature%20Pledge%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655101 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kaz_flag.gif?h=da9490b2&amp;itok=6gGhFnLE" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kazakhstan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC_Kazakhstan%203.0%20eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kazakhstan NDC 3.0</a><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC%20Kazakhstan%203.0%20russ.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kazakhstan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC_Kazakhstan%203.0%20eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kazakhstan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-11/NDC%20Kazakhstan%203.0%20russ.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kazakhstan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC_Kazakhstan%203.0%20eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kazakhstan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-11/NDC%20Kazakhstan%203.0%20russ.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kazakhstan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-654987 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/qat_flag.gif?h=75e47415&amp;itok=E8KOovya" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Qatar
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Qatar%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Qatar NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Qatar%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Qatar NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">21/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Qatar%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Qatar NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-654345 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mex_flag.gif?h=27ba19da&amp;itok=ah4BCLpQ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mexico
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Me%CC%81xico_spanish.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Me%CC%81xico_spanish.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Me%CC%81xico_spanish.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-654232 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/yem_flag_0.gif?h=57cf074e&amp;itok=SlwPslYs" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Yemen
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span>(*) </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Yemens%20NDC3.0%20Vision.pdf" class="ndc-acr-download-link is-original" hreflang="en">Yemen NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Yemens%20NDC3.0%20Vision.pdf" class="ndc-acr-download-link is-original" hreflang="en">Yemen NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Yemens%20NDC3.0%20Vision.pdf" class="ndc-acr-download-link is-original" hreflang="en">Yemen NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-654085 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cri_flag.gif?h=27ba19da&amp;itok=iUi-adHk" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Costa Rica
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/CND-2025-2035.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica NDC 2025 - 2035</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CND-2025-2035.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica NDC 2025 - 2035</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">14/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CND-2025-2035.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica NDC 2025 - 2035</a></div>
</td>
</tr>
<tr class="submission-nid-653770 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/irq_flag_0.gif?h=2ad7dc73&amp;itok=6LgLlzR9" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Iraq
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">Arabic</span><a href="https://unfccc.int/sites/default/files/2025-11/%D9%88%D8%AB%D9%8A%D9%82%D8%A9%20%D8%A7%D9%84%D9%85%D8%B3%D8%A7%D9%87%D9%85%D8%A7%D8%AA%20%D8%A7%D9%84%D9%85%D8%AD%D8%AF%D8%AF%D8%A9%20%D9%88%D8%B7%D9%86%D9%8A%D8%A7%20-%202025.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Iraq NDC 3.0</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2026-01/NDC%20Report%20EN%20-%202025.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Iraq NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">Arabic</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/%D9%88%D8%AB%D9%8A%D9%82%D8%A9%20%D8%A7%D9%84%D9%85%D8%B3%D8%A7%D9%87%D9%85%D8%A7%D8%AA%20%D8%A7%D9%84%D9%85%D8%AD%D8%AF%D8%AF%D8%A9%20%D9%88%D8%B7%D9%86%D9%8A%D8%A7%20-%202025.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Iraq NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2026-01/NDC%20Report%20EN%20-%202025.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Iraq NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/%D9%88%D8%AB%D9%8A%D9%82%D8%A9%20%D8%A7%D9%84%D9%85%D8%B3%D8%A7%D9%87%D9%85%D8%A7%D8%AA%20%D8%A7%D9%84%D9%85%D8%AD%D8%AF%D8%AF%D8%A9%20%D9%88%D8%B7%D9%86%D9%8A%D8%A7%20-%202025.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Iraq NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2026-01/NDC%20Report%20EN%20-%202025.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Iraq NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-653502 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/dji_flag.gif?h=f3674879&amp;itok=tRV1VqIr" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Djibouti
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-11/CDN%20revisee_Djibouti_Novembre%202025.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN%20revisee_Djibouti_Novembre%202025.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN%20revisee_Djibouti_Novembre%202025.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-653298 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ukr_flag.gif?h=d1720097&amp;itok=Rjq1ZHq3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ukraine
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 42px;"><a href="https://unfccc.int/sites/default/files/2025-11/2%20%D0%9D%D0%92%D0%922%20%D0%BF%D1%80%D0%BE%D1%94%D0%BA%D1%82%20%28%D0%B7%D0%BC%D1%96%D0%BD%D0%B5%D0%BD%D0%B0%20%D1%86%D1%96%D0%BB%D1%8C%20_%20%D0%B7%D0%B2%D1%96%D1%82%29%20.pdf" class="ndc-acr-download-link is-original">Ukraine Second NDC</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2025-11/2%20Ukraine%20NDC2_adj_v2.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Ukraine Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 42px;"></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/2%20%D0%9D%D0%92%D0%922%20%D0%BF%D1%80%D0%BE%D1%94%D0%BA%D1%82%20%28%D0%B7%D0%BC%D1%96%D0%BD%D0%B5%D0%BD%D0%B0%20%D1%86%D1%96%D0%BB%D1%8C%20_%20%D0%B7%D0%B2%D1%96%D1%82%29%20.pdf" class="ndc-acr-download-link is-original">Ukraine Second NDC</a><a href="https://unfccc.int/sites/default/files/2025-11/2%20Ukraine%20NDC2_adj_v2.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Ukraine Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">11/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/2%20%D0%9D%D0%92%D0%922%20%D0%BF%D1%80%D0%BE%D1%94%D0%BA%D1%82%20%28%D0%B7%D0%BC%D1%96%D0%BD%D0%B5%D0%BD%D0%B0%20%D1%86%D1%96%D0%BB%D1%8C%20_%20%D0%B7%D0%B2%D1%96%D1%82%29%20.pdf" class="ndc-acr-download-link is-original">Ukraine Second NDC</a><a href="https://unfccc.int/sites/default/files/2025-11/2%20Ukraine%20NDC2_adj_v2.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Ukraine Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-653157 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bdi_flag.gif?h=57cf074e&amp;itok=M1gP3ooO" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Burundi
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-11/CDN3.0%20%20BURUNDI.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN3.0%20%20BURUNDI.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN3.0%20%20BURUNDI.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-653146 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/btn_flag.gif?h=57cf074e&amp;itok=3EUOB0qp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bhutan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Third%20NDC%20%28Provisional%29_10%20November%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan's NDC 3.0 (Provisional)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Third%20NDC%20%28Provisional%29_10%20November%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan's NDC 3.0 (Provisional)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Third%20NDC%20%28Provisional%29_10%20November%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan's NDC 3.0 (Provisional)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-653022 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/blr_flag_0.gif?h=da9490b2&amp;itok=qg9XIK2J" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Belarus
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/2025-11/Republic%20of%20Belarus%20NDC%20for%202026-2035.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Republic of Belarus NDC for 2026-2035</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Russian</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Republic%20of%20Belarus%20NDC%20for%202026-2035.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Republic of Belarus NDC for 2026-2035</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Republic%20of%20Belarus%20NDC%20for%202026-2035.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Republic of Belarus NDC for 2026-2035</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-652920 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tur_flag.gif?h=57cf074e&amp;itok=Bsjl49a2" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Türkiye
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/The%20Second%20NDC%20of%20T%C3%BCrkiye.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Türkiye Second NDC (NDC 3.0)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/The%20Second%20NDC%20of%20T%C3%BCrkiye.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Türkiye Second NDC (NDC 3.0)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_0">0</span><span class="alt_0 ndc_submission_version_0">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/The%20Second%20NDC%20of%20T%C3%BCrkiye.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Türkiye Second NDC (NDC 3.0)</a></div>
</td>
</tr>
<tr class="submission-nid-652811 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bhs_flag.gif?h=da9490b2&amp;itok=T2Q_8_YY" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bahamas
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/The%20Bahamas%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahamas NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/The%20Bahamas%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahamas NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">07/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/The%20Bahamas%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahamas NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
</td>
</tr>
<tr class="submission-nid-652813 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pry_flag.gif?h=da9490b2&amp;itok=xSwda_X3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Paraguay
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0%20%28Anexo%20t%C3%A9cnico%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC Technical Annex</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0%20%28Anexo%20t%C3%A9cnico%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC Technical Annex</a></div>
</td>
</tr>
<tr class="submission-nid-652263 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/per_flag.gif?h=57cf074e&amp;itok=qorWi1PB" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Peru
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/Documento%20NDC%203.0_UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru’s Updated Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Documento%20NDC%203.0_UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru’s Updated Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Documento%20NDC%203.0_UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru’s Updated Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
</tr>
<tr class="submission-nid-652279 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/uzb_flag.gif?h=da9490b2&amp;itok=Uk1GSfcz" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Uzbekistan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC%20rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan NDC 3.0</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC%20rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC%20rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-652301 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gin_flag.gif?h=57cf074e&amp;itok=R4VpVcwp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Guinea
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-11/CDN%203.0%20DE%20LA%20REPUBLIQUE%20DE%20GUINEE.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN%203.0%20DE%20LA%20REPUBLIQUE%20DE%20GUINEE.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN%203.0%20DE%20LA%20REPUBLIQUE%20DE%20GUINEE.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-652810 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fji_flag.gif?h=da9490b2&amp;itok=x6GQCrGd" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Fiji
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Melanesia" data-region-id="5917"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Fiji%20NDC3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Fiji%20NDC3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Fiji%20NDC3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-652941 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cyp_flag_0.gif?h=27ba19da&amp;itok=BvfXakal" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cyprus
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652942 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cze_flag_0.gif?h=bfb37f4a&amp;itok=KTP99jan" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Czechia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652940 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hrv_flag.gif?h=b650b131&amp;itok=yw1I8PNh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Croatia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652945 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fin_flag.gif?h=d1720097&amp;itok=1364Ux9S" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Finland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652943 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/dnk_flag.gif?h=c3f562c8&amp;itok=-uSGdQls" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Denmark
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652944 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/est_flag.gif?h=d1720097&amp;itok=Ehy_GjOJ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Estonia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-651964 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/moz_flag.gif?h=d4b38aaf&amp;itok=BzqOJiNF" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mozambique
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Mozambique%20ProvNDC_ENG.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique's Provisional NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Mozambique%20ProvNDC_ENG.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique's Provisional NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-11/Submission_Letter_Mozambique.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission Letter Mozambique</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Mozambique%20ProvNDC_ENG.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique's Provisional NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-11/Submission_Letter_Mozambique.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission Letter Mozambique</a></div>
</td>
</tr>
<tr class="submission-nid-652939 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bgr_flag_0.gif?h=57cf074e&amp;itok=HsdbUL2X" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bulgaria
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652938 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bel_flag.gif?h=37efeadd&amp;itok=_0IpmWAW" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Belgium
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652937 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/aut_flag.gif?h=f64c9c91&amp;itok=3qcsMW-w" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Austria
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652217 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/aze_flag_0.gif?h=da9490b2&amp;itok=orSCTF7_" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Azerbaijan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Report_Azerbaijan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Azerbaijan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Report_Azerbaijan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Azerbaijan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Report_Azerbaijan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Azerbaijan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-652041 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/eu_flag.gif?h=57cf074e&amp;itok=-JZLp2wh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
European Union (EU)
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 209px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en"> The nationally determined contribution of the European Union and its Member States (EU NDC)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 209px;"><span class="field--name-field-set-item-language is-original" style="height: 209px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en"> The nationally determined contribution of the European Union and its Member States (EU NDC)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en"> The nationally determined contribution of the European Union and its Member States (EU NDC)</a></div>
</td>
</tr>
<tr class="submission-nid-652955 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lux_flag.gif?h=27ba19da&amp;itok=2TqxIE2q" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Luxembourg
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652947 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/germany-162301_960_720_0.png?h=1c7500b8&amp;itok=aiALI4zU" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Germany
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-654226 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sur_flag.gif?h=57cf074e&amp;itok=wWHdlhA1" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Suriname
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203%20report%20Suriname%20251104%20Final%20Publication%20Version.pdf" class="ndc-acr-download-link is-original" hreflang="en">Suriname NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203%20report%20Suriname%20251104%20Final%20Publication%20Version.pdf" class="ndc-acr-download-link is-original" hreflang="en">Suriname NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203%20report%20Suriname%20251104%20Final%20Publication%20Version.pdf" class="ndc-acr-download-link is-original" hreflang="en">Suriname NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-652966 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/swe_flag.gif?h=d4b38aaf&amp;itok=_ISgfkla" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sweden
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652965 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/esp_flag.gif?h=57cf074e&amp;itok=RD0eEf-K" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Spain
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652964 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/svn_flag.gif?h=da9490b2&amp;itok=DsruIlLj" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Slovenia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652961 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/svk_flag.gif?h=57cf074e&amp;itok=2kq8g1LL" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Slovakia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652960 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/rom_flag.gif?h=57cf074e&amp;itok=q9Lh3Azg" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Romania
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652959 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/prt_flag.gif?h=57cf074e&amp;itok=dpFQi-4p" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Portugal
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652958 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pol_flag_1.gif?h=cd032e74&amp;itok=es2H90SR" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Poland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652957 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nld_flag.gif?h=57cf074e&amp;itok=nl3ou0Tg" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Netherlands
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652956 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mlt_flag.gif?h=d4b38aaf&amp;itok=s-nGsWHa" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Malta
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652954 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ltu_flag.gif?h=da9490b2&amp;itok=9FpT5aly" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Lithuania
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652953 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lva_flag.gif?h=da9490b2&amp;itok=qwa90wmO" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Latvia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652952 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ita_flag.gif?h=57cf074e&amp;itok=jYtFVF_s" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Italy
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652951 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/irl_flag.gif?h=da9490b2&amp;itok=KLKAozv9" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ireland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652950 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hun_flag.gif?h=57cf074e&amp;itok=lGk5gntI" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Hungary
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652948 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/grc_flag.gif?h=2ad7dc73&amp;itok=nadtxlB6" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Greece
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652946 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fra_flag.gif?h=57cf074e&amp;itok=FwVIPVaY" width="57" height="35" alt="" typeof="Image" class="img-responsive">
France
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-01/20250106-Composante-PTOM-CDN-FR_vF.pdf" class="ndc-acr-download-link is-original">Composante PTOM de la CDN Française</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-01/20250106-Composante-PTOM-CDN-FR_vF.pdf" class="ndc-acr-download-link is-original">Composante PTOM de la CDN Française</a></div>
</td>
</tr>
<tr class="submission-nid-651855 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tha_flag.gif?h=57cf074e&amp;itok=uE0sncE0" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Thailand
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/TH%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Thailand​ NDC 3.0</a></div>


</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>


</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/TH%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Thailand​ NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">04/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/TH%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Thailand​ NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-651325 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/chn_flag_0.gif?h=57cf074e&amp;itok=bO2AXDMj" width="57" height="35" alt="" typeof="Image" class="img-responsive">
China
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Chinese</span><a href="https://unfccc.int/sites/default/files/2025-11/2035%E5%B9%B4%E4%B8%AD%E5%9B%BD%E5%9B%BD%E5%AE%B6%E8%87%AA%E4%B8%BB%E8%B4%A1%E7%8C%AE%E6%8A%A5%E5%91%8A.pdf" class="ndc-acr-download-link is-original" hreflang="zh">China’s 2035 National Determined Contributions </a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Chinese</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/2035%E5%B9%B4%E4%B8%AD%E5%9B%BD%E5%9B%BD%E5%AE%B6%E8%87%AA%E4%B8%BB%E8%B4%A1%E7%8C%AE%E6%8A%A5%E5%91%8A.pdf" class="ndc-acr-download-link is-original" hreflang="zh">China’s 2035 National Determined Contributions </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">03/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/2035%E5%B9%B4%E4%B8%AD%E5%9B%BD%E5%9B%BD%E5%AE%B6%E8%87%AA%E4%B8%BB%E8%B4%A1%E7%8C%AE%E6%8A%A5%E5%91%8A.pdf" class="ndc-acr-download-link is-original" hreflang="zh">China’s 2035 National Determined Contributions </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-651324 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cpv_flag.gif?h=e57e762b&amp;itok=OiYxshzb" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cabo Verde
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Cabo%20Verde.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cabo Verde NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Cabo%20Verde.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cabo Verde NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">01/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Cabo%20Verde.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cabo Verde NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650920 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/civ_flag_0.gif?h=2ad7dc73&amp;itok=QJ5kq-Ll" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Côte d'Ivoire
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-10/CDN%203.0%20COTE%20D%27IVOIRE.pdf" class="ndc-acr-download-link is-original" hreflang="fr"> Côte D'Ivoire NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/CDN%203.0%20COTE%20D%27IVOIRE.pdf" class="ndc-acr-download-link is-original" hreflang="fr"> Côte D'Ivoire NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/10/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/CDN%203.0%20COTE%20D%27IVOIRE.pdf" class="ndc-acr-download-link is-original" hreflang="fr"> Côte D'Ivoire NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650901 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/Flag_of_Mauritania.png?h=1c7500b8&amp;itok=yQvcTVsG" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mauritania
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-10/CDN3.0_Mauritanie_Signed.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mauritania NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/CDN3.0_Mauritanie_Signed.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mauritania NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">28/10/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/CDN3.0_Mauritanie_Signed.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mauritania NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650888 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ven_flag.gif?h=57cf074e&amp;itok=SuBAO7n_" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Venezuela (Bolivarian Republic of)
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-10/Segunda%20Contribuci%C3%B3n%20Determinada%20Nacional%20CDN%202025-2030.pdf" class="ndc-acr-download-link is-original" hreflang="es">Venezuela (Bolivarian Republic of) Second NDC </a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Segunda%20Contribuci%C3%B3n%20Determinada%20Nacional%20CDN%202025-2030.pdf" class="ndc-acr-download-link is-original" hreflang="es">Venezuela (Bolivarian Republic of) Second NDC </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-10/Nota%20Verbal%20N%C2%B0%20136%20-%20Entrega%20de%20la%20contribuci%C3%B3n%20nacionalmente%20determinada%20de%20Venezuela%20%282025%E2%80%932030%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">NOTA VERBAL</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">28/10/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Segunda%20Contribuci%C3%B3n%20Determinada%20Nacional%20CDN%202025-2030.pdf" class="ndc-acr-download-link is-original" hreflang="es">Venezuela (Bolivarian Republic of) Second NDC </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-10/Nota%20Verbal%20N%C2%B0%20136%20-%20Entrega%20de%20la%20contribuci%C3%B3n%20nacionalmente%20determinada%20de%20Venezuela%20%282025%E2%80%932030%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">NOTA VERBAL</a></div>
</td>
</tr>
<tr class="submission-nid-650669 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/idn_flag.gif?h=2f1c05c4&amp;itok=-Yrphy9G" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Indonesia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-10/Indonesia_Second%20NDC_2025.10.24.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic Of Indonesia Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Indonesia_Second%20NDC_2025.10.24.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic Of Indonesia Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/10/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Indonesia_Second%20NDC_2025.10.24.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic Of Indonesia Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-650607 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/zaf_flag.gif?h=57cf074e&amp;itok=hmgwDH56" width="57" height="35" alt="" typeof="Image" class="img-responsive">
South Africa
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Southern Africa" data-region-id="5899"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-10/South%20Africa%27s%20second%20Nationally%20Determined%20Contribution_2025.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Africa's Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/South%20Africa%27s%20second%20Nationally%20Determined%20Contribution_2025.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Africa's Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/10/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/South%20Africa%27s%20second%20Nationally%20Determined%20Contribution_2025.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Africa's Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-650604 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mys_flag.gif?h=b650b131&amp;itok=yiTRQvdW" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Malaysia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-10/Malaysia%20NDC%203.0%20to%20UNFCCC%202025%20final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malaysia NDC 3.0 2025 </a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Malaysia%20NDC%203.0%20to%20UNFCCC%202025%20final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malaysia NDC 3.0 2025 </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/10/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Malaysia%20NDC%203.0%20to%20UNFCCC%202025%20final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malaysia NDC 3.0 2025 </a></div>
</td>
</tr>
<tr class="submission-nid-650219 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kgz_flag.gif?h=e57e762b&amp;itok=-hm6Wlad" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kyrgyzstan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/2025-10/NDC3.0_Kyrgyzstan_Russian_30-09-2025.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kyrgyzstan NDC 3.0</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2025-10/NDC3.0_Kyrgyzstan_English_30-09-2025%20%282%29.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kyrgyzstan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/NDC3.0_Kyrgyzstan_Russian_30-09-2025.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kyrgyzstan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-10/NDC3.0_Kyrgyzstan_English_30-09-2025%20%282%29.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kyrgyzstan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">02/10/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/NDC3.0_Kyrgyzstan_Russian_30-09-2025.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kyrgyzstan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-10/NDC3.0_Kyrgyzstan_English_30-09-2025%20%282%29.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kyrgyzstan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650174 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/isl_flag.gif?h=a745b187&amp;itok=6D6YcEH1" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Iceland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Iceland%27s%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland‘s NDC 2035</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Iceland%27s%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland‘s NDC 2035</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Iceland%27s%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland‘s NDC 2035</a></div>
</td>
</tr>
<tr class="submission-nid-650168 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lbn_flag.gif?h=2ad7dc73&amp;itok=MurLgFTB" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Lebanon
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/LBN%20NDC%203.0%2030.09.2025%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lebanon NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/LBN%20NDC%203.0%2030.09.2025%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lebanon NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC%203.0%20UNFCCC%20Submission%20Sept2025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Letter of Submission</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/LBN%20NDC%203.0%2030.09.2025%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lebanon NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC%203.0%20UNFCCC%20Submission%20Sept2025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Letter of Submission</a></div>
</td>
</tr>
<tr class="submission-nid-650146 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/Seychelles%20flag.jpg?h=c1beacee&amp;itok=ZZfRDObb" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Seychelles
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/ICTU%20Update%202025%20for%20Seychelles%20new%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Seychelles NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/ICTU%20Update%202025%20for%20Seychelles%20new%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Seychelles NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/ICTU%20Update%202025%20for%20Seychelles%20new%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Seychelles NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650173 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/stp_flag.gif?h=da9490b2&amp;itok=GwE_lpL9" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sao Tome and Principe
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/NDC3.0_Sao_Tome_Principe_F.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sao Tome and Principe NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC3.0_Sao_Tome_Principe_F.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sao Tome and Principe NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC3.0_Sao_Tome_Principe_F.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sao Tome and Principe NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650184 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pan_flag.gif?h=57cf074e&amp;itok=QSPIis5C" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Panama
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-10/Declaratoria%20-%20CDN%203.0%20Panam%C3%A1.pdf" class="ndc-acr-download-link is-original" hreflang="es">Declarative NDC 3.0 of Panama</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Declaratoria%20-%20CDN%203.0%20Panam%C3%A1.pdf" class="ndc-acr-download-link is-original" hreflang="es">Declarative NDC 3.0 of Panama</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Declaratoria%20-%20CDN%203.0%20Panam%C3%A1.pdf" class="ndc-acr-download-link is-original" hreflang="es">Declarative NDC 3.0 of Panama</a></div>
</td>
</tr>
<tr class="submission-nid-650183 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mar_flag.gif?h=57cf074e&amp;itok=KZ58SgW7" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Morocco
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Northern Africa" data-region-id="5896"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-10/MOROCCO%20NDC%203.0%20_30.9.25.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Morocco NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/MOROCCO%20NDC%203.0%20_30.9.25.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Morocco NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/MOROCCO%20NDC%203.0%20_30.9.25.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Morocco NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650203 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/vct_flag.gif?h=57cf074e&amp;itok=6Df69H8G" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Saint Vincent and the Grenadines
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-10/Revised%20NDC%202.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Vincent and the Grenadines NDC 2.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Revised%20NDC%202.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Vincent and the Grenadines NDC 2.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/Revised%20NDC%202.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Vincent and the Grenadines NDC 2.0</a></div>
</td>
</tr>
<tr class="submission-nid-649837 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bol_flag_0_0.gif?h=57cf074e&amp;itok=3ACVx1LM" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bolivia (Plurinational State of)
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-09/Bolivia_NDC3.0_2026-2035%20final%2029.09.2024.pdf" class="ndc-acr-download-link is-original" hreflang="es">Bolivia (Plurinational State of) NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Bolivia_NDC3.0_2026-2035%20final%2029.09.2024.pdf" class="ndc-acr-download-link is-original" hreflang="es">Bolivia (Plurinational State of) NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Bolivia_NDC3.0_2026-2035%20final%2029.09.2024.pdf" class="ndc-acr-download-link is-original" hreflang="es">Bolivia (Plurinational State of) NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650141 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/rus_flag.gif?h=57cf074e&amp;itok=XREo3dah" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Russian Federation
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/2025-09/%D0%A0%D0%A4%20%D0%B2%D1%82%D0%BE%D1%80%D0%BE%D0%B9%20%D0%9E%D0%9D%D0%A3%D0%92.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Russian Federation Second NDC</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2025-09/RF_second_NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Russian Federation Second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Russian</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/%D0%A0%D0%A4%20%D0%B2%D1%82%D0%BE%D1%80%D0%BE%D0%B9%20%D0%9E%D0%9D%D0%A3%D0%92.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Russian Federation Second NDC</a><a href="https://unfccc.int/sites/default/files/2025-09/RF_second_NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Russian Federation Second NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/%D0%A0%D0%A4%20%D0%B2%D1%82%D0%BE%D1%80%D0%BE%D0%B9%20%D0%9E%D0%9D%D0%A3%D0%92.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Russian Federation Second NDC</a><a href="https://unfccc.int/sites/default/files/2025-09/RF_second_NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Russian Federation Second NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-650128 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bgd_flag.gif?h=27ba19da&amp;itok=2NLSJc3i" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bangladesh
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Bangladesh%20Third%20Nationally%20Determined%20Contribution%20%28NDC%203.0%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bangladesh NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Bangladesh%20Third%20Nationally%20Determined%20Contribution%20%28NDC%203.0%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bangladesh NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Bangladesh%20Third%20Nationally%20Determined%20Contribution%20%28NDC%203.0%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bangladesh NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650117 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mus_flag.gif?h=57cf074e&amp;itok=dENyD122" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mauritius
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/NDC%203.0%20%20Mauritius.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mauritius NDC 3.0 </a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC%203.0%20%20Mauritius.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mauritius NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/cover%20letter%20NDC%203.0%20Mauritius.pdf" class="ndc-acr-download-link is-original">Mauritius Cover Letter</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC%203.0%20%20Mauritius.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mauritius NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/cover%20letter%20NDC%203.0%20Mauritius.pdf" class="ndc-acr-download-link is-original">Mauritius Cover Letter</a></div>
</td>
</tr>
<tr class="submission-nid-650100 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/eth_flag.gif?h=da9490b2&amp;itok=uZITk-Yc" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ethiopia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Ethiopia%20NDC%203.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ethiopia NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Ethiopia%20NDC%203.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ethiopia NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Ethiopia%20NDC%203.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ethiopia NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650099 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tuv_flag.gif?h=da9490b2&amp;itok=18ucAI7V" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Tuvalu
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Tuvalu%20NDC%203.0%20-%20Secretariat%20Submission%20FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tuvalu NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Tuvalu%20NDC%203.0%20-%20Secretariat%20Submission%20FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tuvalu NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Tuvalu%20NDC%203.0%20-%20Secretariat%20Submission%20FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tuvalu NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-650098 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lka_flag.gif?h=da9490b2&amp;itok=eTx_IqFv" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sri Lanka
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Sri%20Lankas%20Nationally%20Determined%20Contributions%203.0%20%282026-2035%29%20submitted%2022.09.2025%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sri Lanka NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Sri%20Lankas%20Nationally%20Determined%20Contributions%203.0%20%282026-2035%29%20submitted%2022.09.2025%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sri Lanka NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">25/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Sri%20Lankas%20Nationally%20Determined%20Contributions%203.0%20%282026-2035%29%20submitted%2022.09.2025%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sri Lanka NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-649756 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/isl_flag.gif?h=a745b187&amp;itok=6D6YcEH1" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Iceland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Icelands%20NDC%202030%20-%20Revision_1.1.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland‘s NDC 2030 – Revision</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Icelands%20NDC%202030%20-%20Revision_1.1.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland‘s NDC 2030 – Revision</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">25/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Icelands%20NDC%202030%20-%20Revision_1.1.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland‘s NDC 2030 – Revision</a></div>
</td>
</tr>
<tr class="submission-nid-649980 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lbr_flag.gif?h=6c49f853&amp;itok=cFEXbMjA" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Liberia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Liberias_2035_NDC_3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liberia NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Liberias_2035_NDC_3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liberia NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Liberias_2035_NDC_3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liberia NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649979 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pak_flag.gif?h=57cf074e&amp;itok=mxnIht9Y" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Pakistan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Pakistan_NDC3.0_24%20Sep.pdf" class="ndc-acr-download-link is-original" hreflang="en">Pakistan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Pakistan_NDC3.0_24%20Sep.pdf" class="ndc-acr-download-link is-original" hreflang="en">Pakistan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Pakistan_NDC3.0_24%20Sep.pdf" class="ndc-acr-download-link is-original" hreflang="en">Pakistan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649972 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fsm_flag.gif?h=57cf074e&amp;itok=v3cvSY85" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Micronesia (Federated States of)
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/FSM%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Federated States of Micronesia NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/FSM%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Federated States of Micronesia NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/FSM%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Federated States of Micronesia NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649971 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/vut_flag.gif?h=27ba19da&amp;itok=790xdzce" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Vanuatu
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Melanesia" data-region-id="5917"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Vanuatu%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vanuatu NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Vanuatu%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vanuatu NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Vanuatu%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vanuatu NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649970 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mng_flag.gif?h=da9490b2&amp;itok=hi3Nm8CI" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mongolia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Mongolia%20NDC3_0%20under%20UNFCCC_PA%20FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mongolia NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Mongolia%20NDC3_0%20under%20UNFCCC_PA%20FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mongolia NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Mongolia%20NDC3_0%20under%20UNFCCC_PA%20FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mongolia NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649954 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/chl_flag.gif?h=57cf074e&amp;itok=zAAmLFG_" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Chile
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-09/NDC-2025-220925%201.pdf" class="ndc-acr-download-link is-original" hreflang="es">Chile NDC 2025</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC-2025-220925%201.pdf" class="ndc-acr-download-link is-original" hreflang="es">Chile NDC 2025</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">23/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC-2025-220925%201.pdf" class="ndc-acr-download-link is-original" hreflang="es">Chile NDC 2025</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-649945 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ton_flag.gif?h=da9490b2&amp;itok=KH1tJU-R" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Tonga
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Tongas%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tonga NDC 3.0 </a></div>


</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>


</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Tongas%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tonga NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/Tongas%20Second%20Nationally%20Determined%20Contribution%20Review%20Report.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tongas Second Nationally Determined Contribution Review Report</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">23/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Tongas%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tonga NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/Tongas%20Second%20Nationally%20Determined%20Contribution%20Review%20Report.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tongas Second Nationally Determined Contribution Review Report</a></div>
</td>
</tr>
<tr class="submission-nid-649917 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tun_flag_0.gif?h=57cf074e&amp;itok=-57hHt6F" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Tunisia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Northern Africa" data-region-id="5896"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Republic%20Of%20Tunisia-Draft%20preleminary%20elements%20of%20the%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Preliminary draft element of the new Tunisian NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Republic%20Of%20Tunisia-Draft%20preleminary%20elements%20of%20the%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Preliminary draft element of the new Tunisian NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Republic%20Of%20Tunisia-Draft%20preleminary%20elements%20of%20the%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Preliminary draft element of the new Tunisian NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-649915 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nga_flag.gif?h=da9490b2&amp;itok=CymVuZ3S" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Nigeria
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Nigeria%20NDC%203.0%20-%20Transimission%20Version%202.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nigeria NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Nigeria%20NDC%203.0%20-%20Transimission%20Version%202.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nigeria NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Nigeria%20NDC%203.0%20-%20Transimission%20Version%202.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nigeria NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-649916 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/jor_flag.gif?h=da9490b2&amp;itok=kZzHDVDC" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Jordan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Jordan%20NDC%203.0%20Vision_final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jordan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Jordan%20NDC%203.0%20Vision_final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jordan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Jordan%20NDC%203.0%20Vision_final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jordan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649918 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hnd_flag.gif?h=d4b38aaf&amp;itok=R8mDwun-" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Honduras
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-09/Honduras%20Segunda%20Actualizacio%CC%81n%20de%20su%20NDC%20%282021-2030%29serna_rev_final.pdf" class="ndc-acr-download-link is-original" hreflang="es">Segunda Actualización de la Contribución Nacional Determinada (NDC) </a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original" style="height: 167px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Honduras%20Segunda%20Actualizacio%CC%81n%20de%20su%20NDC%20%282021-2030%29serna_rev_final.pdf" class="ndc-acr-download-link is-original" hreflang="es">Segunda Actualización de la Contribución Nacional Determinada (NDC) </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/OFICIO%20141.pdf" class="ndc-acr-download-link is-original" hreflang="es">Forward Note for submission NDC </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Honduras%20Segunda%20Actualizacio%CC%81n%20de%20su%20NDC%20%282021-2030%29serna_rev_final.pdf" class="ndc-acr-download-link is-original" hreflang="es">Segunda Actualización de la Contribución Nacional Determinada (NDC) </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/OFICIO%20141.pdf" class="ndc-acr-download-link is-original" hreflang="es">Forward Note for submission NDC </a></div>
</td>
</tr>
<tr class="submission-nid-649919 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/swz_flag.gif?h=57cf074e&amp;itok=mU4Kq2Vd" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Eswatini
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Southern Africa" data-region-id="5899"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Eswatini%20NDC%203.0%20Report_for%20Submission%20-FINAL%20f.pdf" class="ndc-acr-download-link is-original" hreflang="en">Eswatini NDC 3.0 </a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Eswatini%20NDC%203.0%20Report_for%20Submission%20-FINAL%20f.pdf" class="ndc-acr-download-link is-original" hreflang="en">Eswatini NDC 3.0 </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Eswatini%20NDC%203.0%20Report_for%20Submission%20-FINAL%20f.pdf" class="ndc-acr-download-link is-original" hreflang="en">Eswatini NDC 3.0 </a></div>
</td>
</tr>
<tr class="submission-nid-649944 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/jam_flag.gif?h=b8db4804&amp;itok=ZlnsCAiy" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Jamaica
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Jamaica%20ICTU%20Final%20Report%20Sep19.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jamaica NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Jamaica%20ICTU%20Final%20Report%20Sep19.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jamaica NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-10/NDC_Report_Oct12.pdf" class="ndc-acr-download-link is-original" hreflang="en">Technical Report Supporting Jamaica NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Jamaica%20ICTU%20Final%20Report%20Sep19.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jamaica NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-10/NDC_Report_Oct12.pdf" class="ndc-acr-download-link is-original" hreflang="en">Technical Report Supporting Jamaica NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649876 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nic_flag.gif?h=27ba19da&amp;itok=5OxBWN7b" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Nicaragua
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-09/NDC%20NICARAGUA%202025%20-12.09.2025.pdf" class="ndc-acr-download-link is-original" hreflang="es">Nicaragua NDC 2025</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC%20NICARAGUA%202025%20-12.09.2025.pdf" class="ndc-acr-download-link is-original" hreflang="es">Nicaragua NDC 2025</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC%20NICARAGUA%202025%20-12.09.2025.pdf" class="ndc-acr-download-link is-original" hreflang="es">Nicaragua NDC 2025</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-649857 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/aus_flag.gif?h=b8db4804&amp;itok=2OmNpOBH" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Australia
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Australia and New Zealand" data-region-id="5916"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Australias%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Australia’s second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Australias%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Australia’s second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">18/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Australias%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Australia’s second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-649815 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/480px-Flag_of_the_Vatican_City.svg__0.png?h=593c9b4a&amp;itok=YjbdA5C6" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Holy See
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Holy%20See_NDC%202035_September%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Holy See NDC 2035</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Holy%20See_NDC%202035_September%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Holy See NDC 2035</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Holy%20See_NDC%202035_September%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Holy See NDC 2035</a></div>
</td>
</tr>
<tr class="submission-nid-650902 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/brn_flag_0.gif?h=da9490b2&amp;itok=0N0fzB-l" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Brunei Darussalam
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-10/%5BFinal%5D%20Brunei%20Darussalam%27s%20Nationally%20Determined%20Contribution%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Brunei Darussalam NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/%5BFinal%5D%20Brunei%20Darussalam%27s%20Nationally%20Determined%20Contribution%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Brunei Darussalam NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-10/%5BFinal%5D%20Brunei%20Darussalam%27s%20Nationally%20Determined%20Contribution%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Brunei Darussalam NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-649785 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ago_flag.gif?h=c5aaa1f1&amp;itok=PKHSWfsT" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Angola
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Angola%20NDC_September2025_Upload.pdf" class="ndc-acr-download-link is-original" hreflang="en">Angola NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Angola%20NDC_September2025_Upload.pdf" class="ndc-acr-download-link is-original" hreflang="en">Angola NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Angola%20NDC_September2025_Upload.pdf" class="ndc-acr-download-link is-original" hreflang="en">Angola NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649757 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/srb_flag.gif?h=57cf074e&amp;itok=_d-D4hV4" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Serbia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/NDC3%20of%20the%20Republic%20of%20Serbia.pdf" class="ndc-acr-download-link is-original" hreflang="en">Serbia NDC 3.0 </a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC3%20of%20the%20Republic%20of%20Serbia.pdf" class="ndc-acr-download-link is-original" hreflang="en">Serbia NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/Republic%20of%20Serbia%20NDC3.0%20Official%20Submission%20Letter.pdf" class="ndc-acr-download-link is-original" hreflang="en">NDC Submission Letter Serbia</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/NDC3%20of%20the%20Republic%20of%20Serbia.pdf" class="ndc-acr-download-link is-original" hreflang="en">Serbia NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-09/Republic%20of%20Serbia%20NDC3.0%20Official%20Submission%20Letter.pdf" class="ndc-acr-download-link is-original" hreflang="en">NDC Submission Letter Serbia</a></div>
</td>
</tr>
<tr class="submission-nid-649667 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lie_flag.gif?h=27ba19da&amp;itok=B3CTc8dA" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Liechtenstein
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Liechtensteins%20Second%20NDC%202035_September%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liechtenstein Second NDC Submission </a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Liechtensteins%20Second%20NDC%202035_September%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liechtenstein Second NDC Submission </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Liechtensteins%20Second%20NDC%202035_September%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liechtenstein Second NDC Submission </a></div>
</td>
</tr>
<tr class="submission-nid-649598 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/som_flag.gif?h=57cf074e&amp;itok=Pjehj8Kk" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Somalia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Somalia%20NDC%203.0_Official_2025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated Somalia's NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Somalia%20NDC%203.0_Official_2025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated Somalia's NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/09/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Somalia%20NDC%203.0_Official_2025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated Somalia's NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649406 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/brb_flag.gif?h=bfb37f4a&amp;itok=_XIege06" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Barbados
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Barbados%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Barbados 2025 Second Nationally Determined Contribution</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Barbados%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Barbados 2025 Second Nationally Determined Contribution</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/08/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Barbados%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Barbados 2025 Second Nationally Determined Contribution</a></div>
</td>
</tr>
<tr class="submission-nid-649205 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/slb_flag.gif?h=27ee3080&amp;itok=8BeGXdi3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Solomon Islands
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Melanesia" data-region-id="5917"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-09/Solomon%20Islands%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Solomon Islands NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Solomon%20Islands%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Solomon Islands NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/08/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-09/Solomon%20Islands%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Solomon Islands NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-649146 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/khm_flag_0.gif?h=d1720097&amp;itok=E8F1KKT5" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cambodia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-08/Cambodia-NDC%203.0_0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cambodia’s NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-08/Cambodia-NDC%203.0_0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cambodia’s NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/08/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-08/Cambodia-NDC%203.0_0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cambodia’s NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-648832 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/niu_flag.gif?h=da9490b2&amp;itok=cEebG22u" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Niue
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-07/NIUE%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Niue NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-07/NIUE%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Niue NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/07/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-07/NIUE%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Niue NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-648825 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mco_flag.gif?h=d4b38aaf&amp;itok=EgWt5YfP" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Monaco
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-07/NDC_2025_Monaco.pdf" class="ndc-acr-download-link is-original" hreflang="fr">MONACO NDC 2025</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-07/NDC_2025_Monaco.pdf" class="ndc-acr-download-link is-original" hreflang="fr">MONACO NDC 2025</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/07/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-07/NDC_2025_Monaco.pdf" class="ndc-acr-download-link is-original" hreflang="fr">MONACO NDC 2025</a></div>
</td>
</tr>
<tr class="submission-nid-648558 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nor_flag.gif?h=dc257391&amp;itok=1jiEHiIZ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Norway
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-06/Norways%20NDC%20for%202035..pdf" class="ndc-acr-download-link is-original" hreflang="en">Norway NDC 2035</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-06/Norways%20NDC%20for%202035..pdf" class="ndc-acr-download-link is-original" hreflang="en">Norway NDC 2035</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/06/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-06/Norways%20NDC%20for%202035..pdf" class="ndc-acr-download-link is-original" hreflang="en">Norway NDC 2035</a></div>
</td>
</tr>
<tr class="submission-nid-647614 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/blz_flag.gif?h=57cf074e&amp;itok=b6a37X0i" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Belize
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-06/BELIZE%20FINAL%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Belize NDC 3.0 Update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-06/BELIZE%20FINAL%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Belize NDC 3.0 Update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">14/06/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-06/BELIZE%20FINAL%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Belize NDC 3.0 Update</a></div>
</td>
</tr>
<tr class="submission-nid-646931 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/npl_flag.gif?h=3ffbe2c7&amp;itok=LNckKNR-" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Nepal
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-05/Nepal%20NDC3.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nationally Determined Contribution (NDC) 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-05/Nepal%20NDC3.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nationally Determined Contribution (NDC) 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/05/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-05/Nepal%20NDC3.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nationally Determined Contribution (NDC) 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-646682 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mda_flag.gif?h=da9490b2&amp;itok=P38ufCeK" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Republic of Moldova
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-05/MD_NDC_3.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Moldova NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-05/MD_NDC_3.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Moldova NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/05/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-05/MD_NDC_3.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Moldova NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-646649 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ken_flag.gif?h=57cf074e&amp;itok=DKa8bTUE" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kenya
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-05/KENYAS%20SECOND%20NATIONALLY%20DETERMINED%20CONTRIBUTION%202031_2035.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kenya's Second Nationally Determined Contribution 2031 - 2035</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-05/KENYAS%20SECOND%20NATIONALLY%20DETERMINED%20CONTRIBUTION%202031_2035.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kenya's Second Nationally Determined Contribution 2031 - 2035</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/04/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-05/KENYAS%20SECOND%20NATIONALLY%20DETERMINED%20CONTRIBUTION%202031_2035.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kenya's Second Nationally Determined Contribution 2031 - 2035</a></div>
</td>
</tr>
<tr class="submission-nid-645866 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cub_flag_0.gif?h=da9490b2&amp;itok=e9yt4jRh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cuba
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-02/REPUBLICA%20DE%20CUBA%20CND3.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Cuba NDC 3.0 </a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/REPUBLICA%20DE%20CUBA%20CND3.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Cuba NDC 3.0 </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/REPUBLICA%20DE%20CUBA%20CND3.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Cuba NDC 3.0 </a></div>
</td>
</tr>
<tr class="submission-nid-645864 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/120px-Maldives_flag_300.png?h=bfb37f4a&amp;itok=ZX15KCRc" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Maldives
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/Maldives%E2%80%99%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Maldives' Third Nationally Determined Contribution </a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Maldives%E2%80%99%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Maldives' Third Nationally Determined Contribution </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Maldives%E2%80%99%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Maldives' Third Nationally Determined Contribution </a></div>
</td>
</tr>
<tr class="submission-nid-645826 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mne_flag.gif?h=d72776d6&amp;itok=NcrBvS3f" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Montenegro
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/001_eng_NDC_Montenegro.pdf" class="ndc-acr-download-link is-original" hreflang="en">Montenegro Third NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/001_eng_NDC_Montenegro.pdf" class="ndc-acr-download-link is-original" hreflang="en">Montenegro Third NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">21/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/001_eng_NDC_Montenegro.pdf" class="ndc-acr-download-link is-original" hreflang="en">Montenegro Third NDC</a></div>
</td>
</tr>
<tr class="submission-nid-645764 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/jpn_flag.gif?h=c5aaa1f1&amp;itok=36Zi5tJZ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Japan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/Japans%202035-2040%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Japan's 2035/2040 NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Japans%202035-2040%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Japan's 2035/2040 NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">18/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Japans%202035-2040%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Japan's 2035/2040 NDC</a></div>
</td>
</tr>
<tr class="submission-nid-645660 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/can_flag.gif?h=b8db4804&amp;itok=mpdXI6mS" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Canada
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Northern America" data-region-id="5903"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/Canada%27s%202035%20Nationally%20Determined%20Contribution_ENc.pdf" class="ndc-acr-download-link is-original" hreflang="en">Canada's 2035 NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-02/Soumission%20officielle%20de%20la%20CDN%20du%20Canada%20-%20CCNUCC%20v2fr.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Canada's 2035 NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Canada%27s%202035%20Nationally%20Determined%20Contribution_ENc.pdf" class="ndc-acr-download-link is-original" hreflang="en">Canada's 2035 NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Soumission%20officielle%20de%20la%20CDN%20du%20Canada%20-%20CCNUCC%20v2fr.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Canada's 2035 NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Canada%27s%202035%20Nationally%20Determined%20Contribution_ENc.pdf" class="ndc-acr-download-link is-original" hreflang="en">Canada's 2035 NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Soumission%20officielle%20de%20la%20CDN%20du%20Canada%20-%20CCNUCC%20v2fr.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Canada's 2035 NDC</a></div>
</td>
</tr>
<tr class="submission-nid-645646 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/zwe_flag.gif?h=b8db4804&amp;itok=-0MzRI73" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Zimbabwe
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/Zimbabwe%20NDC3.0%20Country%20Statement_2025_35.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zimbabwe NDC3.0 Country Statement</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Zimbabwe%20NDC3.0%20Country%20Statement_2025_35.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zimbabwe NDC3.0 Country Statement</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Zimbabwe%20NDC3.0%20Country%20Statement_2025_35.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zimbabwe NDC3.0 Country Statement</a></div>
</td>
</tr>
<tr class="submission-nid-645633 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mhl_flag.gif?h=27ee3080&amp;itok=dv-Jy3ab" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Marshall Islands
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/Republic%20of%20the%20Marshall%20Islands%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Marshall Islands Third NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Republic%20of%20the%20Marshall%20Islands%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Marshall Islands Third NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Republic%20of%20the%20Marshall%20Islands%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Marshall Islands Third NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-645632 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sgp_flag.gif?h=57cf074e&amp;itok=QiJuFZdu" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Singapore
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/Singapore%20Second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Singapore Second Nationally Determined Contribution</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Singapore%20Second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Singapore Second Nationally Determined Contribution</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_4">4</span><span class="alt_0 ndc_submission_version_4">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Singapore%20Second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Singapore Second Nationally Determined Contribution</a></div>
</td>
</tr>
<tr class="submission-nid-645603 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ecu_flag.gif?h=da9490b2&amp;itok=TUBnN1Bu" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ecuador
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-02/Segunda%20NDC%20de%20Ecuador.pdf" class="ndc-acr-download-link is-original" hreflang="es">Ecuador Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Segunda%20NDC%20de%20Ecuador.pdf" class="ndc-acr-download-link is-original" hreflang="es">Ecuador Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Segunda%20NDC%20de%20Ecuador.pdf" class="ndc-acr-download-link is-original" hreflang="es">Ecuador Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-645604 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lca_flag.gif?h=da9490b2&amp;itok=NyOJfrvn" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Saint Lucia
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/Saint%20Lucias%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Lucia Third NDC </a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Saint%20Lucias%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Lucia Third NDC </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Saint%20Lucias%20Third%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Lucia Third NDC </a></div>
</td>
</tr>
<tr class="submission-nid-645602 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lso_flag.gif?h=57cf074e&amp;itok=006wqWlu" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Lesotho
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Southern Africa" data-region-id="5899"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-02/Lesotho%20SECOND%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lesotho Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Lesotho%20SECOND%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lesotho Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/Lesotho%20SECOND%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lesotho Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-645600 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/and_flag.gif?h=4cee71d3&amp;itok=k3dmFt9Y" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Andorra
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-02/NDC%203.0%20ANDORRA.pdf" class="ndc-acr-download-link is-original" hreflang="es">Andorra Third NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/NDC%203.0%20ANDORRA.pdf" class="ndc-acr-download-link is-original" hreflang="es">Andorra Third NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/02/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-02/NDC%203.0%20ANDORRA.pdf" class="ndc-acr-download-link is-original" hreflang="es">Andorra Third NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-645548 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nzl_flag.gif?h=da9490b2&amp;itok=8tBGQTt7" width="57" height="35" alt="" typeof="Image" class="img-responsive">
New Zealand
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Australia and New Zealand" data-region-id="5916"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-01/New%20Zealand%27s%20second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">New Zealand’s second Nationally Determined Contribution</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-01/New%20Zealand%27s%20second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">New Zealand’s second Nationally Determined Contribution</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/01/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-01/New%20Zealand%27s%20second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">New Zealand’s second Nationally Determined Contribution</a></div>
</td>
</tr>
<tr class="submission-nid-645546 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gbr_flag.gif?h=b8db4804&amp;itok=V-eMDajL" width="57" height="35" alt="" typeof="Image" class="img-responsive">
United Kingdom of Great Britain and Northern Ireland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-01/UK%27s%202035%20NDC%20ICTU.pdf" class="ndc-acr-download-link is-original" hreflang="en">The UK’s 2035 NDC ICTU</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-01/UK%27s%202035%20NDC%20ICTU.pdf" class="ndc-acr-download-link is-original" hreflang="en">The UK’s 2035 NDC ICTU</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/01/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-01/UK%27s%202035%20NDC%20ICTU.pdf" class="ndc-acr-download-link is-original" hreflang="en">The UK’s 2035 NDC ICTU</a></div>
</td>
</tr>
<tr class="submission-nid-645541 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/che_flag.gif?h=3ea932e1&amp;itok=T6r-frTk" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Switzerland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-01/Switzerland%20second%20NDC%202031-2035.pdf" class="ndc-acr-download-link is-original" hreflang="en">Switzerland second NDC 2031-2035</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-01/Annex%20to%20Switzerland%20NDC%202031-2035.pdf" class="ndc-acr-download-link is-original" hreflang="en">Annex to Switzerland NDC 2031-2035</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-01/Switzerland%20second%20NDC%202031-2035.pdf" class="ndc-acr-download-link is-original" hreflang="en">Switzerland second NDC 2031-2035</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/01/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-01/Annex%20to%20Switzerland%20NDC%202031-2035.pdf" class="ndc-acr-download-link is-original" hreflang="en">Annex to Switzerland NDC 2031-2035</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-01/Switzerland%20second%20NDC%202031-2035.pdf" class="ndc-acr-download-link is-original" hreflang="en">Switzerland second NDC 2031-2035</a></div>
</td>
</tr>
<tr class="submission-nid-645369 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ury_flag.gif?h=57cf074e&amp;itok=g5beHMdE" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Uruguay
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-01/20241220_Uruguay_NDC3.pdf" class="ndc-acr-download-link is-original" hreflang="es">Uruguay Third NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-01/20241220_Uruguay_NDC3.pdf" class="ndc-acr-download-link is-original" hreflang="es">Uruguay Third NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/12/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-01/20241220_Uruguay_NDC3.pdf" class="ndc-acr-download-link is-original" hreflang="es">Uruguay Third NDC</a></div>
</td>
</tr>
<tr class="submission-nid-645073 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bwa_flag_0.gif?h=57cf074e&amp;itok=FRTB4C3o" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Botswana
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Southern Africa" data-region-id="5899"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 188px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2024-12/BOTSWANA_NDC_2%20REPORT.pdf" class="ndc-acr-download-link is-original" hreflang="en">Botswana’s 2nd Updated Nationally Determined Contribution under the Paris Agreement</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 188px;"><span class="field--name-field-set-item-language is-original" style="height: 188px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-12/BOTSWANA_NDC_2%20REPORT.pdf" class="ndc-acr-download-link is-original" hreflang="en">Botswana’s 2nd Updated Nationally Determined Contribution under the Paris Agreement</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/12/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-12/BOTSWANA_NDC_2%20REPORT.pdf" class="ndc-acr-download-link is-original" hreflang="en">Botswana’s 2nd Updated Nationally Determined Contribution under the Paris Agreement</a></div>
</td>
</tr>
<tr class="submission-nid-645000 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/usa_flag.gif?h=27ee3080&amp;itok=1dBodVbS" width="57" height="35" alt="" typeof="Image" class="img-responsive">
United States of America
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Northern America" data-region-id="5903"></span>(*) </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2024-12/United%20States%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">United States of America 2035 NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-12/United%20States%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">United States of America 2035 NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/12/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-12/United%20States%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">United States of America 2035 NDC</a></div>
</td>
</tr>
<tr class="submission-nid-643655 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/che_flag.gif?h=3ea932e1&amp;itok=T6r-frTk" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Switzerland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2024-11/Switzerlands%20First%20NDC_2021_2030_Update%202024_including%20ICTUs.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Switzerland First NDC (2021-2030 Update 2024 including ICTUs)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-11/Switzerlands%20First%20NDC_2021_2030_Update%202024_including%20ICTUs.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Switzerland First NDC (2021-2030 Update 2024 including ICTUs)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_4">4</span><span class="alt_0 ndc_submission_version_4">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">14/11/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-11/Switzerlands%20First%20NDC_2021_2030_Update%202024_including%20ICTUs.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Switzerland First NDC (2021-2030 Update 2024 including ICTUs)</a></div>
</td>
</tr>
<tr class="submission-nid-643338 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bra_flag_0.gif?h=4cee71d3&amp;itok=cCNVVxQF" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Brazil
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2024-11/Brazil_Second%20Nationally%20Determined%20Contribution%20%28NDC%29_November2024.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brazil Second Nationally Determined Contribution</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-11/Brazil_Second%20Nationally%20Determined%20Contribution%20%28NDC%29_November2024.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brazil Second Nationally Determined Contribution</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/11/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-11/Brazil_Second%20Nationally%20Determined%20Contribution%20%28NDC%29_November2024.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brazil Second Nationally Determined Contribution</a></div>
</td>
</tr>
<tr class="submission-nid-642248 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/are_flag.gif?h=da9490b2&amp;itok=qLtK5Vww" width="57" height="35" alt="" typeof="Image" class="img-responsive">
United Arab Emirates
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2024-11/UAE-NDC3.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">The United Arab Emirates’ Third Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original" style="height: 167px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-11/UAE-NDC3.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">The United Arab Emirates’ Third Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2024-11/UAE-NDC3.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">The United Arab Emirates’ Third Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
</tr>
<tr class="submission-nid-639823 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pan_flag.gif?h=57cf074e&amp;itok=QSPIis5C" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Panama
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2024-06/Segunda%20Contribuci%C3%B3n%20Determinada%20a%20Nivel%20Nacional_CDN2.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama Second NDC </a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2024-06/Segunda%20Contribuci%C3%B3n%20Determinada%20a%20Nivel%20Nacional_CDN2.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama Second NDC </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/06/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2024-06/Segunda%20Contribuci%C3%B3n%20Determinada%20a%20Nivel%20Nacional_CDN2.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama Second NDC </a></div>
</td>
</tr>
<tr class="submission-nid-636851 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mdg_flag.gif?h=57cf074e&amp;itok=3DXFYRyH" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Madagascar
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2024-01/NDC%202%20MADAGASCAR.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC 2 Madagascar</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2024-01/NDC%202%20MADAGASCAR.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC 2 Madagascar</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_0">0</span><span class="alt_0 ndc_submission_version_0">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/01/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2024-01/NDC%202%20MADAGASCAR.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC 2 Madagascar</a></div>
</td>
</tr>
<tr class="submission-nid-636780 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nam_flag.gif?h=57cf074e&amp;itok=LpyQ4J-I" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Namibia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Southern Africa" data-region-id="5899"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2024-01/FINAL%20UPDATED%20NAMIBIA%20NDC%202023.pdf" class="ndc-acr-download-link is-original" hreflang="en">NDC UPDATE</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2024-01/FINAL%20UPDATED%20NAMIBIA%20NDC%202023.pdf" class="ndc-acr-download-link is-original" hreflang="en">NDC UPDATE</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/01/2024 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2024-01/FINAL%20UPDATED%20NAMIBIA%20NDC%202023.pdf" class="ndc-acr-download-link is-original" hreflang="en">NDC UPDATE</a></div>
</td>
</tr>
<tr class="submission-nid-634359 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/omn_flag.gif?h=57cf074e&amp;itok=t5rR3Jf5" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Oman
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-11/Oman%201st%20Update%20of%20the%202nd%20NDC%20-%20Optimized%20Size%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Oman 1st Update of the 2nd NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-11/Oman%201st%20Update%20of%20the%202nd%20NDC%20-%20Optimized%20Size%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Oman 1st Update of the 2nd NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/11/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-11/Oman%201st%20Update%20of%20the%202nd%20NDC%20-%20Optimized%20Size%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Oman 1st Update of the 2nd NDC</a></div>
</td>
</tr>
<tr class="submission-nid-633023 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bra_flag_0.gif?h=4cee71d3&amp;itok=cCNVVxQF" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Brazil
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-11/Brazil%20First%20NDC%202023%20adjustment.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brazil First NDC 2023 adjustment</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-11/Brazil%20First%20NDC%202023%20adjustment.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brazil First NDC 2023 adjustment</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_4">4</span><span class="alt_0 ndc_submission_version_4">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">03/11/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-11/Brazil%20First%20NDC%202023%20adjustment.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brazil First NDC 2023 adjustment</a></div>
</td>
</tr>
<tr class="submission-nid-632613 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/grc_flag.gif?h=2ad7dc73&amp;itok=nadtxlB6" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Greece
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632611 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/germany-162301_960_720_0.png?h=1c7500b8&amp;itok=aiALI4zU" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Germany
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632615 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hun_flag.gif?h=57cf074e&amp;itok=lGk5gntI" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Hungary
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632617 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/irl_flag.gif?h=da9490b2&amp;itok=KLKAozv9" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ireland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632619 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ita_flag.gif?h=57cf074e&amp;itok=jYtFVF_s" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Italy
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632621 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lva_flag.gif?h=da9490b2&amp;itok=qwa90wmO" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Latvia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632623 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ltu_flag.gif?h=da9490b2&amp;itok=9FpT5aly" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Lithuania
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632625 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lux_flag.gif?h=27ba19da&amp;itok=2TqxIE2q" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Luxembourg
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632627 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mlt_flag.gif?h=d4b38aaf&amp;itok=s-nGsWHa" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Malta
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632629 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nld_flag.gif?h=57cf074e&amp;itok=nl3ou0Tg" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Netherlands
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632631 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pol_flag_1.gif?h=cd032e74&amp;itok=es2H90SR" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Poland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632633 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/prt_flag.gif?h=57cf074e&amp;itok=dpFQi-4p" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Portugal
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632635 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/rom_flag.gif?h=57cf074e&amp;itok=q9Lh3Azg" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Romania
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632637 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/svk_flag.gif?h=57cf074e&amp;itok=2kq8g1LL" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Slovakia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632639 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/svn_flag.gif?h=da9490b2&amp;itok=DsruIlLj" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Slovenia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632641 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/esp_flag.gif?h=57cf074e&amp;itok=RD0eEf-K" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Spain
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632643 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/swe_flag.gif?h=d4b38aaf&amp;itok=_ISgfkla" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sweden
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632609 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fra_flag.gif?h=57cf074e&amp;itok=FwVIPVaY" width="57" height="35" alt="" typeof="Image" class="img-responsive">
France
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/FR%20CDN%20addendum%20r%C3%A9vis%C3%A9%20-%202021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">FR CDN addendum révisé - 2021</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/FR%20CDN%20addendum%20r%C3%A9vis%C3%A9%20-%202021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">FR CDN addendum révisé - 2021</a></div>
</td>
</tr>
<tr class="submission-nid-632607 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fin_flag.gif?h=d1720097&amp;itok=1364Ux9S" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Finland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632605 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/est_flag.gif?h=d1720097&amp;itok=Ehy_GjOJ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Estonia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632603 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/dnk_flag.gif?h=c3f562c8&amp;itok=-uSGdQls" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Denmark
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632601 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cze_flag_0.gif?h=bfb37f4a&amp;itok=KTP99jan" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Czechia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632599 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cyp_flag_0.gif?h=27ba19da&amp;itok=BvfXakal" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cyprus
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632597 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hrv_flag.gif?h=b650b131&amp;itok=yw1I8PNh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Croatia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632595 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bgr_flag_0.gif?h=57cf074e&amp;itok=HsdbUL2X" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bulgaria
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632593 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bel_flag.gif?h=37efeadd&amp;itok=_0IpmWAW" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Belgium
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632591 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/aut_flag.gif?h=f64c9c91&amp;itok=3qcsMW-w" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Austria
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632584 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/eu_flag.gif?h=57cf074e&amp;itok=-JZLp2wh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
European Union (EU)
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/ES-2023-10-17%20EU%20submission%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">EU NDC 2023 update</a></div>
</td>
</tr>
<tr class="submission-nid-632447 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/aze_flag_0.gif?h=da9490b2&amp;itok=orSCTF7_" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Azerbaijan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-10/Second%20NDC_Azerbaijan_ENG_Final%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated Nationally Determined Contribution of the Republic of Azerbaijan</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original" style="height: 167px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/Second%20NDC_Azerbaijan_ENG_Final%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated Nationally Determined Contribution of the Republic of Azerbaijan</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/10/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-10/Second%20NDC_Azerbaijan_ENG_Final%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated Nationally Determined Contribution of the Republic of Azerbaijan</a></div>
</td>
</tr>
<tr class="submission-nid-630388 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kaz_flag.gif?h=da9490b2&amp;itok=6gGhFnLE" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kazakhstan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-06/12updated%20NDC%20KAZ_Gov%20Decree313_19042023_en_cover%20page.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kazakhstan First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-06/12updated%20NDC%20KAZ_Gov%20Decree313_19042023_en_cover%20page.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kazakhstan First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/06/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-06/12updated%20NDC%20KAZ_Gov%20Decree313_19042023_en_cover%20page.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kazakhstan First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-630377 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/egy_flag.gif?h=57cf074e&amp;itok=SEt7XyIH" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Egypt
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Northern Africa" data-region-id="5896"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-06/Egypts%20Updated%20First%20Nationally%20Determined%20Contribution%202030%20%28Second%20Update%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Egypt’s Updated First Nationally Determined Contribution 2030 (Second Update)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original" style="height: 167px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-06/Egypts%20Updated%20First%20Nationally%20Determined%20Contribution%202030%20%28Second%20Update%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Egypt’s Updated First Nationally Determined Contribution 2030 (Second Update)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/06/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-06/Egypts%20Updated%20First%20Nationally%20Determined%20Contribution%202030%20%28Second%20Update%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Egypt’s Updated First Nationally Determined Contribution 2030 (Second Update)</a></div>
</td>
</tr>
<tr class="submission-nid-628703 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/480px-Flag_of_the_Vatican_City.svg__0.png?h=593c9b4a&amp;itok=YjbdA5C6" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Holy See
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-05/Vatican%20City%20State%20NDCs%20-%20May%202023.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vatican City State's NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-05/Vatican%20City%20State%20NDCs%20-%20May%202023.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vatican City State's NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/05/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-05/Vatican%20City%20State%20NDCs%20-%20May%202023.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vatican City State's NDC</a></div>
</td>
</tr>
<tr class="submission-nid-627744 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tur_flag.gif?h=57cf074e&amp;itok=Bsjl49a2" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Türkiye
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-04/T%C3%9CRK%C4%B0YE_UPDATED%201st%20NDC_EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">TÜRKİYE UPDATED 1st NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-04/T%C3%9CRK%C4%B0YE_UPDATED%201st%20NDC_EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">TÜRKİYE UPDATED 1st NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_0">0</span><span class="alt_0 ndc_submission_version_0">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/04/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-04/T%C3%9CRK%C4%B0YE_UPDATED%201st%20NDC_EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">TÜRKİYE UPDATED 1st NDC</a></div>
</td>
</tr>
<tr class="submission-nid-627081 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kir_flag.gif?h=27ba19da&amp;itok=HesmOwUS" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kiribati
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-03/221213%20Kiribati%20NDC%20Web%20Quality.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kiribati Enhanced NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-03/221213%20Kiribati%20NDC%20Web%20Quality.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kiribati Enhanced NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">02/03/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-03/221213%20Kiribati%20NDC%20Web%20Quality.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kiribati Enhanced NDC</a></div>
</td>
</tr>
<tr class="submission-nid-625900 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tkm_flag.gif?h=da9490b2&amp;itok=UoIi0RBp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Turkmenistan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 188px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-01/NDC_Turkmenistan_12-05-2022_approv.%20by%20Decree_Eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Nationally Determined Contribution of Turkmenistan under the Paris Agreement</a><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/NDC/2023-01/NDC_Turkmenistan_12-05-2022_approv.%20by%20Decree_Rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Nationally Determined Contribution of Turkmenistan under the Paris Agreement</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 188px;"><span class="field--name-field-set-item-language is-original" style="height: 188px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-01/NDC_Turkmenistan_12-05-2022_approv.%20by%20Decree_Eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Nationally Determined Contribution of Turkmenistan under the Paris Agreement</a><a href="https://unfccc.int/sites/default/files/NDC/2023-01/NDC_Turkmenistan_12-05-2022_approv.%20by%20Decree_Rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Nationally Determined Contribution of Turkmenistan under the Paris Agreement</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/01/2023 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-01/NDC_Turkmenistan_12-05-2022_approv.%20by%20Decree_Eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Nationally Determined Contribution of Turkmenistan under the Paris Agreement</a><a href="https://unfccc.int/sites/default/files/NDC/2023-01/NDC_Turkmenistan_12-05-2022_approv.%20by%20Decree_Rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Nationally Determined Contribution of Turkmenistan under the Paris Agreement</a></div>
</td>
</tr>
<tr class="submission-nid-624765 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ury_flag.gif?h=57cf074e&amp;itok=g5beHMdE" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Uruguay
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-12/Uruguay%20Segunda%20CDN.pdf" class="ndc-acr-download-link is-original" hreflang="es">Uruguay Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-12/Uruguay%20Segunda%20CDN.pdf" class="ndc-acr-download-link is-original" hreflang="es">Uruguay Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/12/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-12/Uruguay%20Segunda%20CDN.pdf" class="ndc-acr-download-link is-original" hreflang="es">Uruguay Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-624283 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mex_flag.gif?h=27ba19da&amp;itok=ah4BCLpQ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mexico
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Mexico_NDC_UNFCCC_update2022_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico: Updated NDC 2022</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Mexico_NDC_UNFCCC_update2022_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico: Updated NDC 2022</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Mexico_NDC_UNFCCC_update2022_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico: Updated NDC 2022</a></div>
</td>
</tr>
<tr class="submission-nid-626589 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tuv_flag.gif?h=da9490b2&amp;itok=18ucAI7V" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Tuvalu
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-02/Tuvalus%20Updated%20NDC%20for%20UNFCCC%20Submission.pdf" class="ndc-acr-download-link is-original" hreflang="en">Government of Tuvalu Updated Nationally Determined Contribution</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-02/Tuvalus%20Updated%20NDC%20for%20UNFCCC%20Submission.pdf" class="ndc-acr-download-link is-original" hreflang="en">Government of Tuvalu Updated Nationally Determined Contribution</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-02/Tuvalus%20Updated%20NDC%20for%20UNFCCC%20Submission.pdf" class="ndc-acr-download-link is-original" hreflang="en">Government of Tuvalu Updated Nationally Determined Contribution</a></div>
</td>
</tr>
<tr class="submission-nid-622542 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/vnm_flag.gif?h=57cf074e&amp;itok=c-wpfKR7" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Viet Nam
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Viet%20Nam%20NDC%202022%20Update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Viet Nam NDC 2022 Update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Viet%20Nam%20NDC%202022%20Update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Viet Nam NDC 2022 Update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Viet%20Nam%20NDC%202022%20Update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Viet Nam NDC 2022 Update</a></div>
</td>
</tr>
<tr class="submission-nid-622332 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tls_flag.gif?h=d4b38aaf&amp;itok=21izQCBk" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Timor-Leste
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Timor_Leste%20Updated%20NDC%202022_2030.pdf" class="ndc-acr-download-link is-original" hreflang="en">Timor-Leste Updated NDC 2022-2030</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Timor_Leste%20Updated%20NDC%202022_2030.pdf" class="ndc-acr-download-link is-original" hreflang="en">Timor-Leste Updated NDC 2022-2030</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Timor_Leste%20Updated%20NDC%202022_2030.pdf" class="ndc-acr-download-link is-original" hreflang="en">Timor-Leste Updated NDC 2022-2030</a></div>
</td>
</tr>
<tr class="submission-nid-622330 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/and_flag.gif?h=4cee71d3&amp;itok=k3dmFt9Y" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Andorra
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2023-01/221125_Actualizaci%C3%B3n_NDC_DEF.pdf" class="ndc-acr-download-link is-original" hreflang="es">Andorra 2022 NDC Update</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-01/221125_Actualizaci%C3%B3n_NDC_DEF.pdf" class="ndc-acr-download-link is-original" hreflang="es">Andorra 2022 NDC Update</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/20222410_Actualitzacio%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Andorra 2022 NDC Update (Draft)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-01/221125_Actualizaci%C3%B3n_NDC_DEF.pdf" class="ndc-acr-download-link is-original" hreflang="es">Andorra 2022 NDC Update</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/20222410_Actualitzacio%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Andorra 2022 NDC Update (Draft)</a></div>
</td>
</tr>
<tr class="submission-nid-622032 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bhs_flag.gif?h=da9490b2&amp;itok=T2Q_8_YY" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bahamas
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Bahamas%20Updated%20Nationally%20Determined%20Contributio" class="ndc-acr-download-link is-original" hreflang="en">Bahamas Updated Nationally Determined Contributions, 2022</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Bahamas%20Updated%20Nationally%20Determined%20Contributio" class="ndc-acr-download-link is-original" hreflang="en">Bahamas Updated Nationally Determined Contributions, 2022</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">07/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Bahamas%20Updated%20Nationally%20Determined%20Contributio" class="ndc-acr-download-link is-original" hreflang="en">Bahamas Updated Nationally Determined Contributions, 2022</a></div>
</td>
</tr>
<tr class="submission-nid-621292 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sgp_flag.gif?h=57cf074e&amp;itok=QiJuFZdu" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Singapore
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Singapore%20Second%20Update%20of%20First%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Second Update of First Nationally Determined Contribution</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Singapore%20Second%20Update%20of%20First%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Second Update of First Nationally Determined Contribution</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">04/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Singapore%20Second%20Update%20of%20First%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Second Update of First Nationally Determined Contribution</a></div>
</td>
</tr>
<tr class="submission-nid-620976 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nor_flag.gif?h=dc257391&amp;itok=1jiEHiIZ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Norway
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-11/NDC%20Norway_second%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Norway First NDC (Second updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/NDC%20Norway_second%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Norway First NDC (Second updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">03/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/NDC%20Norway_second%20update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Norway First NDC (Second updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-620604 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tha_flag.gif?h=57cf074e&amp;itok=uE0sncE0" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Thailand
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Thailand%202nd%20Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Thailand​ 2nd​ Updated​ NDC</a></div>


</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>


</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Thailand%202nd%20Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Thailand​ 2nd​ Updated​ NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-01/Cover%20Letter%20-%202nd%20Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Thailand 2nd Updated NDC Letter of Submission</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">02/11/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Thailand%202nd%20Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Thailand​ 2nd​ Updated​ NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2023-01/Cover%20Letter%20-%202nd%20Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Thailand 2nd Updated NDC Letter of Submission</a></div>
</td>
</tr>
<tr class="submission-nid-620168 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gnq_flag.gif?h=57cf074e&amp;itok=H1WdiXEv" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Equatorial Guinea
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-10/CND-GuineaEcuatorial-Version2022-Actualizada.pdf" class="ndc-acr-download-link is-original" hreflang="es">Equatorial Guinea NDC 2022 Update</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-10/CND-GuineaEcuatorial-Version2022-Actualizada.pdf" class="ndc-acr-download-link is-original" hreflang="es">Equatorial Guinea NDC 2022 Update</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/10/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-10/CND-GuineaEcuatorial-Version2022-Actualizada.pdf" class="ndc-acr-download-link is-original" hreflang="es">Equatorial Guinea NDC 2022 Update</a></div>
</td>
</tr>
<tr class="submission-nid-618945 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fsm_flag.gif?h=57cf074e&amp;itok=v3cvSY85" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Micronesia (Federated States of)
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 188px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-10/Updated%20NDC%20of%20the%20MICRONESIA.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Updated Nationally Determined Contribution of the Federated States of Micronesia</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 188px;"><span class="field--name-field-set-item-language is-original" style="height: 188px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-10/Updated%20NDC%20of%20the%20MICRONESIA.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Updated Nationally Determined Contribution of the Federated States of Micronesia</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/10/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-10/Updated%20NDC%20of%20the%20MICRONESIA.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Updated Nationally Determined Contribution of the Federated States of Micronesia</a></div>
</td>
</tr>
<tr class="submission-nid-615083 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/idn_flag.gif?h=2f1c05c4&amp;itok=-Yrphy9G" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Indonesia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-09/ENDC%20Indonesia.pdf" class="ndc-acr-download-link is-original" hreflang="en">Enhanced NDC - Republic of Indonesia</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-09/S335_Indonesia%20Submission%20on%20the%20Enhanced%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Enhanced NDC Cover Letter </a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-09/ENDC%20Indonesia.pdf" class="ndc-acr-download-link is-original" hreflang="en">Enhanced NDC - Republic of Indonesia</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">23/09/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-09/S335_Indonesia%20Submission%20on%20the%20Enhanced%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Enhanced NDC Cover Letter </a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-09/ENDC%20Indonesia.pdf" class="ndc-acr-download-link is-original" hreflang="en">Enhanced NDC - Republic of Indonesia</a></div>
</td>
</tr>
<tr class="submission-nid-616500 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sdn_flag.gif?h=b8db4804&amp;itok=uWfjFYoW" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sudan
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Northern Africa" data-region-id="5896"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-10/Sudan%20Updated%20First%20NDC-12102021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sudan's First NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-10/Sudan%20Updated%20First%20NDC-12102021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sudan's First NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-10/Submission%20of%20Sudan%20Updated%20First%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission of Sudan's First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/09/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-10/Sudan%20Updated%20First%20NDC-12102021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sudan's First NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-10/Submission%20of%20Sudan%20Updated%20First%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission of Sudan's First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-614853 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gbr_flag.gif?h=b8db4804&amp;itok=V-eMDajL" width="57" height="35" alt="" typeof="Image" class="img-responsive">
United Kingdom of Great Britain and Northern Ireland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-09/UK%20NDC%20ICTU%202022.pdf" class="ndc-acr-download-link is-original" hreflang="en">United Kingdom of Great Britain and Northern Ireland updated 2030 NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original" style="height: 167px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-09/UK%20NDC%20ICTU%202022.pdf" class="ndc-acr-download-link is-original" hreflang="en">United Kingdom of Great Britain and Northern Ireland updated 2030 NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/09/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-09/UK%20NDC%20ICTU%202022.pdf" class="ndc-acr-download-link is-original" hreflang="en">United Kingdom of Great Britain and Northern Ireland updated 2030 NDC</a></div>
</td>
</tr>
<tr class="submission-nid-613828 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/uga_flag.gif?h=57cf074e&amp;itok=jGxunB3l" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Uganda
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-09/Updated%20NDC%20_Uganda_2022%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Uganda's Updated NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-09/Updated%20NDC%20_Uganda_2022%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Uganda's Updated NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/09/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-09/Updated%20NDC%20_Uganda_2022%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Uganda's Updated NDC</a></div>
</td>
</tr>
<tr class="submission-nid-611412 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ind_flag.gif?h=57cf074e&amp;itok=rvT2Huv8" width="57" height="35" alt="" typeof="Image" class="img-responsive">
India
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-08/India%20Updated%20First%20Nationally%20Determined%20Contrib.pdf" class="ndc-acr-download-link is-original" hreflang="en">India Updated First Nationally Determined Contribution</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/India%20Updated%20First%20Nationally%20Determined%20Contrib.pdf" class="ndc-acr-download-link is-original" hreflang="en">India Updated First Nationally Determined Contribution</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/Cover%20letter%20from%20Minister%20of%20Environment%20Forest.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cover letter from Minister of Environment Forest</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/08/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/India%20Updated%20First%20Nationally%20Determined%20Contrib.pdf" class="ndc-acr-download-link is-original" hreflang="en">India Updated First Nationally Determined Contribution</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/Cover%20letter%20from%20Minister%20of%20Environment%20Forest.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cover letter from Minister of Environment Forest</a></div>
</td>
</tr>
<tr class="submission-nid-611308 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/srb_flag.gif?h=57cf074e&amp;itok=_d-D4hV4" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Serbia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-08/NDC%20Final_Serbia%20english.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated NDC Serbia</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/NDC%20Final_Serbia%20english.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated NDC Serbia</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/UNFCCC%20Letter%20NDC%20Submission.pdf" class="ndc-acr-download-link is-original" hreflang="en">NDC Submission Letter Serbia</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/08/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/NDC%20Final_Serbia%20english.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated NDC Serbia</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/UNFCCC%20Letter%20NDC%20Submission.pdf" class="ndc-acr-download-link is-original" hreflang="en">NDC Submission Letter Serbia</a></div>
</td>
</tr>
<tr class="submission-nid-578783 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/vut_flag.gif?h=27ba19da&amp;itok=790xdzce" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Vanuatu
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Melanesia" data-region-id="5917"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-08/Vanuatu%20NDC%20Revised%20and%20Enhanced.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vanuatu NDC Revised and Enhanced</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/Vanuatu%20NDC%20Revised%20and%20Enhanced.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vanuatu NDC Revised and Enhanced</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/08/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/Vanuatu%20NDC%20Revised%20and%20Enhanced.pdf" class="ndc-acr-download-link is-original" hreflang="en">Vanuatu NDC Revised and Enhanced</a></div>
</td>
</tr>
<tr class="submission-nid-519269 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gab_flag.gif?h=d4b38aaf&amp;itok=sL8Y0n4I" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Gabon
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-07/20220706_Gabon_Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Seconde Contribution Déterminée au Niveau National (République Gabonaise)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original" style="height: 167px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-07/20220706_Gabon_Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Seconde Contribution Déterminée au Niveau National (République Gabonaise)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/07/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-07/20220706_Gabon_Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Seconde Contribution Déterminée au Niveau National (République Gabonaise)</a></div>
</td>
</tr>
<tr class="submission-nid-513680 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/dma_flag.gif?h=b650b131&amp;itok=3E2DCO8d" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Dominica
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2022-07/The%20Commonwealth%20of%20Dominica%20updated%20NDC%20July%204%20%2C.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Commonwealth of Dominica Updated National Determined Contribution</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original" style="height: 167px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2022-07/The%20Commonwealth%20of%20Dominica%20updated%20NDC%20July%204%20%2C.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Commonwealth of Dominica Updated National Determined Contribution</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">04/07/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2022-07/The%20Commonwealth%20of%20Dominica%20updated%20NDC%20July%204%20%2C.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Commonwealth of Dominica Updated National Determined Contribution</a></div>
</td>
</tr>
<tr class="submission-nid-510663 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/aus_flag.gif?h=b8db4804&amp;itok=2OmNpOBH" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Australia
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Australia and New Zealand" data-region-id="5916"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Australias%20NDC%20June%202022%20Update%20%283%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Australia NDC 2022 Update</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Australias%20NDC%20June%202022%20Update%20%283%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Australia NDC 2022 Update</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%202022%20Update%20Letter%20to%20UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Australia NDC Letter to UNFCCC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_4">4</span><span class="alt_0 ndc_submission_version_4">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/06/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Australias%20NDC%20June%202022%20Update%20%283%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Australia NDC 2022 Update</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%202022%20Update%20Letter%20to%20UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Australia NDC Letter to UNFCCC</a></div>
</td>
</tr>
<tr class="submission-nid-497577 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hti_flag.gif?h=57cf074e&amp;itok=PRZnhaVy" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Haiti
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revisee%20Haiti%202022.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC revised(Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revisee%20Haiti%202022.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC revised(Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">01/06/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revisee%20Haiti%202022.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC revised(Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-499594 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gtm_flag.gif?h=57cf074e&amp;itok=nwR62_Lx" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Guatemala
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2022-06/NDC%20-%20Guatemala%202021.pdf" class="ndc-acr-download-link is-original" hreflang="es">Contribución Nacionalmente Determinada de Guatemala(Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2022-06/NDC%20-%20Guatemala%202021.pdf" class="ndc-acr-download-link is-original" hreflang="es">Contribución Nacionalmente Determinada de Guatemala(Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">23/05/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2022-06/NDC%20-%20Guatemala%202021.pdf" class="ndc-acr-download-link is-original" hreflang="es">Contribución Nacionalmente Determinada de Guatemala(Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497391 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/civ_flag_0.gif?h=2ad7dc73&amp;itok=QJ5kq-Ll" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Côte d'Ivoire
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_CIV_2022.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC COTE D'IVOIRE SUBMISSION</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_CIV_2022.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC COTE D'IVOIRE SUBMISSION</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/05/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_CIV_2022.pdf" class="ndc-acr-download-link is-original" hreflang="fr">NDC COTE D'IVOIRE SUBMISSION</a></div>
</td>
</tr>
<tr class="submission-nid-497372 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/caf_flag.gif?h=bfb37f4a&amp;itok=pdtlTVUp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Central African Republic
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9e%20RCA.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Central African Republic First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9e%20RCA.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Central African Republic First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/01/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9e%20RCA.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Central African Republic First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-499075 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/slv_flag.gif?h=f3674879&amp;itok=GZlrX9HT" width="57" height="35" alt="" typeof="Image" class="img-responsive">
El Salvador
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/El%20Salvador%20NDC-%20Updated%20Dic.2021.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/El%20Salvador%20NDC-%20Updated%20Dic.2021.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/47.%20DCC-UCC-%20072-2021%20UNFCCC%20Remisi%C3%B3n%20NDC%20ACTUALIZADA%20%20El%20Salvador%20Dic%202021.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador First NDC (Updated submission letter)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">04/01/2022 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/El%20Salvador%20NDC-%20Updated%20Dic.2021.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/47.%20DCC-UCC-%20072-2021%20UNFCCC%20Remisi%C3%B3n%20NDC%20ACTUALIZADA%20%20El%20Salvador%20Dic%202021.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador First NDC (Updated submission letter)</a></div>
</td>
</tr>
<tr class="submission-nid-497408 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cdr_flag.gif?h=dc666b08&amp;itok=R38rccWm" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Democratic Republic of the Congo
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9e%20de%20la%20RDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Democratic Republic of the Congo First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9e%20de%20la%20RDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Democratic Republic of the Congo First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">28/12/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9e%20de%20la%20RDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Democratic Republic of the Congo First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497761 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/moz_flag.gif?h=d4b38aaf&amp;itok=BzqOJiNF" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mozambique
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_EN_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_EN_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Submission%20%20UPDATE%20%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique First NDC (Updated submission-letter)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/12/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_EN_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Submission%20%20UPDATE%20%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique First NDC (Updated submission-letter)</a></div>
</td>
</tr>
<tr class="submission-nid-497642 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kor_flag.gif?h=57cf074e&amp;itok=TMP5hws2" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Republic of Korea
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/211223_The%20Republic%20of%20Korea%27s%20Enhanced%20Update%20of%20its%20First%20Nationally%20Determined%20Contribution_211227_editorial%20change.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Korea First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/211223_The%20Republic%20of%20Korea%27s%20Enhanced%20Update%20of%20its%20First%20Nationally%20Determined%20Contribution_211227_editorial%20change.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Korea First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">23/12/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/211223_The%20Republic%20of%20Korea%27s%20Enhanced%20Update%20of%20its%20First%20Nationally%20Determined%20Contribution_211227_editorial%20change.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Korea First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497782 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ner_flag.gif?h=d1720097&amp;itok=_rPSQG7z" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Niger
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_Niger_R%C3%A9vis%C3%A9e_2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Niger First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_Niger_R%C3%A9vis%C3%A9e_2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Niger First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/12/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_Niger_R%C3%A9vis%C3%A9e_2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Niger First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-498035 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ven_flag.gif?h=57cf074e&amp;itok=SuBAO7n_" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Venezuela (Bolivarian Republic of)
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Actualizacion%20NDC%20Venezuela.pdf" class="ndc-acr-download-link is-original" hreflang="es">Venezuela (Bolivarian Republic of) First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Actualizacion%20NDC%20Venezuela.pdf" class="ndc-acr-download-link is-original" hreflang="es">Venezuela (Bolivarian Republic of) First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NOTA%20VERBAL%20CMNUCC%20NDC%20Actualizada%20Venezuela.pdf" class="ndc-acr-download-link is-original" hreflang="es">NOTA VERBAL CMNUCC NDC Actualizada Venezuela</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/11/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Actualizacion%20NDC%20Venezuela.pdf" class="ndc-acr-download-link is-original" hreflang="es">Venezuela (Bolivarian Republic of) First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NOTA%20VERBAL%20CMNUCC%20NDC%20Actualizada%20Venezuela.pdf" class="ndc-acr-download-link is-original" hreflang="es">NOTA VERBAL CMNUCC NDC Actualizada Venezuela</a></div>
</td>
</tr>
<tr class="submission-nid-497419 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/Flag_of_the_Comoros.png?h=a82b7db3&amp;itok=1D4b3DPU" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Comoros
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_r%C3%A9vis%C3%A9e_Comores_vf.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Comoros First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_r%C3%A9vis%C3%A9e_Comores_vf.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Comoros First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_r%C3%A9vis%C3%A9e_Comores_vf.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Comoros First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497516 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gha_flag.gif?h=57cf074e&amp;itok=jRDr58Gh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ghana
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ghana%27s%20Updated%20Nationally%20Determined%20Contribution%20to%20the%20UNFCCC_2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ghana First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ghana%27s%20Updated%20Nationally%20Determined%20Contribution%20to%20the%20UNFCCC_2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ghana First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">04/11/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ghana%27s%20Updated%20Nationally%20Determined%20Contribution%20to%20the%20UNFCCC_2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ghana First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497819 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nzl_flag.gif?h=da9490b2&amp;itok=8tBGQTt7" width="57" height="35" alt="" typeof="Image" class="img-responsive">
New Zealand
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Australia and New Zealand" data-region-id="5916"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/New%20Zealand%20NDC%20November%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">New Zealand First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/New%20Zealand%20NDC%20November%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">New Zealand First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">03/11/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/New%20Zealand%20NDC%20November%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">New Zealand First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497241 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/arg_flag.gif?h=50c26b29&amp;itok=u3pakW9B" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Argentina
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-05/Actualizacio%CC%81n%20meta%20de%20emisiones%202030.pdf" class="ndc-acr-download-link is-original" hreflang="es">Argentina Second NDC (Updated submission)</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2023-12/Argentinas%20Second%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en"> Argentina Second NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-05/Actualizacio%CC%81n%20meta%20de%20emisiones%202030.pdf" class="ndc-acr-download-link is-original" hreflang="es">Argentina Second NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2023-12/Argentinas%20Second%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en"> Argentina Second NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Argentina_Segunda%20Contribuci%C3%B3n%20Nacional.pdf" class="ndc-acr-download-link is-original" hreflang="es">Argentina Second NDC (Archived)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">02/11/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-05/Actualizacio%CC%81n%20meta%20de%20emisiones%202030.pdf" class="ndc-acr-download-link is-original" hreflang="es">Argentina Second NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2023-12/Argentinas%20Second%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en"> Argentina Second NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Argentina_Segunda%20Contribuci%C3%B3n%20Nacional.pdf" class="ndc-acr-download-link is-original" hreflang="es">Argentina Second NDC (Archived)</a></div>
</td>
</tr>
<tr class="submission-nid-498028 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/uzb_flag.gif?h=da9490b2&amp;itok=Uk1GSfcz" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Uzbekistan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Uzbekistan_Updated%20NDC_2021_EN.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Uzbekistan_Updated%20NDC_2021_RU.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Uzbekistan_Updated%20NDC_2021_EN.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Uzbekistan_Updated%20NDC_2021_RU.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Uzbekistan_Updated%20NDC_2021_EN.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Uzbekistan_Updated%20NDC_2021_RU.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497394 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/chn_flag_0.gif?h=57cf074e&amp;itok=bO2AXDMj" width="57" height="35" alt="" typeof="Image" class="img-responsive">
China
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/China%E2%80%99s%20Achievements%2C%20New%20Goals%20and%20New%20Measures%20for%20Nationally%20Determined%20Contributions.pdf" class="ndc-acr-download-link is-translation" hreflang="en">China First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-original">Chinese</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%E4%B8%AD%E5%9B%BD%E8%90%BD%E5%AE%9E%E5%9B%BD%E5%AE%B6%E8%87%AA%E4%B8%BB%E8%B4%A1%E7%8C%AE%E6%88%90%E6%95%88%E5%92%8C%E6%96%B0%E7%9B%AE%E6%A0%87%E6%96%B0%E4%B8%BE%E6%8E%AA.pdf" class="ndc-acr-download-link is-original" hreflang="zh">China First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Chinese</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/China%E2%80%99s%20Achievements%2C%20New%20Goals%20and%20New%20Measures%20for%20Nationally%20Determined%20Contributions.pdf" class="ndc-acr-download-link is-translation" hreflang="en">China First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%E4%B8%AD%E5%9B%BD%E8%90%BD%E5%AE%9E%E5%9B%BD%E5%AE%B6%E8%87%AA%E4%B8%BB%E8%B4%A1%E7%8C%AE%E6%88%90%E6%95%88%E5%92%8C%E6%96%B0%E7%9B%AE%E6%A0%87%E6%96%B0%E4%B8%BE%E6%8E%AA.pdf" class="ndc-acr-download-link is-original" hreflang="zh">China First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/cover%20letter.pdf" class="ndc-acr-download-link is-original" hreflang="zh">China First NDC (Updated submission letter)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-11/%E4%B8%AD%E5%9B%BD%E8%90%BD%E5%AE%9E%E5%9B%BD%E5%AE%B6%E8%87%AA%E4%B8%BB%E8%B4%A1%E7%8C%AE%E8%BF%9B%E5%B1%95%E6%8A%A5%E5%91%8A%202022.pdf" class="ndc-acr-download-link is-original" hreflang="zh">Progress on the Implementation of China NDC 2022-Chinese</a><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Progress%20of%20China%20NDC%202022.pdf" class="ndc-acr-download-link is-original" hreflang="en">Progress on the Implementation of China NDC 2022</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">28/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/China%E2%80%99s%20Achievements%2C%20New%20Goals%20and%20New%20Measures%20for%20Nationally%20Determined%20Contributions.pdf" class="ndc-acr-download-link is-translation" hreflang="en">China First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%E4%B8%AD%E5%9B%BD%E8%90%BD%E5%AE%9E%E5%9B%BD%E5%AE%B6%E8%87%AA%E4%B8%BB%E8%B4%A1%E7%8C%AE%E6%88%90%E6%95%88%E5%92%8C%E6%96%B0%E7%9B%AE%E6%A0%87%E6%96%B0%E4%B8%BE%E6%8E%AA.pdf" class="ndc-acr-download-link is-original" hreflang="zh">China First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/cover%20letter.pdf" class="ndc-acr-download-link is-original" hreflang="zh">China First NDC (Updated submission letter)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-11/%E4%B8%AD%E5%9B%BD%E8%90%BD%E5%AE%9E%E5%9B%BD%E5%AE%B6%E8%87%AA%E4%B8%BB%E8%B4%A1%E7%8C%AE%E8%BF%9B%E5%B1%95%E6%8A%A5%E5%91%8A%202022.pdf" class="ndc-acr-download-link is-original" hreflang="zh">Progress on the Implementation of China NDC 2022-Chinese</a><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Progress%20of%20China%20NDC%202022.pdf" class="ndc-acr-download-link is-original" hreflang="en">Progress on the Implementation of China NDC 2022</a></div>
</td>
</tr>
<tr class="submission-nid-497628 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kna_flag.gif?h=c220e547&amp;itok=nqHe5Cvu" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Saint Kitts and Nevis
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/St.%20Kitts%20and%20Nevis%20Revised%20NDC_Updated.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Kitts and Nevis First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/St.%20Kitts%20and%20Nevis%20Revised%20NDC_Updated.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Kitts and Nevis First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">25/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/St.%20Kitts%20and%20Nevis%20Revised%20NDC_Updated.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Kitts and Nevis First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497889 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sau_flag.gif?h=57cf074e&amp;itok=ZaKxwcmX" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Saudi Arabia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/resource/202203111154---KSA%20NDC%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/resource/202203111154---KSA%20NDC%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/To%20UNFCCC-%201st%20Updated%20NDC%20Submission%20letter.PDF" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">23/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/resource/202203111154---KSA%20NDC%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/To%20UNFCCC-%201st%20Updated%20NDC%20Submission%20letter.PDF" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497615 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/jpn_flag.gif?h=c5aaa1f1&amp;itok=36Zi5tJZ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Japan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/JAPAN_FIRST%20NDC%20%28UPDATED%20SUBMISSION%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Japan First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/JAPAN_FIRST%20NDC%20%28UPDATED%20SUBMISSION%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Japan First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_4">4</span><span class="alt_0 ndc_submission_version_4">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/JAPAN_FIRST%20NDC%20%28UPDATED%20SUBMISSION%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Japan First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497823 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pak_flag.gif?h=57cf074e&amp;itok=mxnIht9Y" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Pakistan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Pakistan%20Updated%20NDC%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Pakistan First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Pakistan%20Updated%20NDC%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Pakistan First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">21/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Pakistan%20Updated%20NDC%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Pakistan First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-499020 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tcd_flag.gif?h=57cf074e&amp;itok=FAhGbDQD" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Chad
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20ACTUALISEE%20DU%20TCHAD.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Chad First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20ACTUALISEE%20DU%20TCHAD.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Chad First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20ACTUALISEE%20DU%20TCHAD.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Chad First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497332 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bhr_flag.gif?h=d4b38aaf&amp;itok=cJULjQL0" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bahrain
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20of%20the%20Kingdom%20of%20Bahrain%20under%20UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20of%20the%20Kingdom%20of%20Bahrain%20under%20UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">18/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20of%20the%20Kingdom%20of%20Bahrain%20under%20UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497570 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/irq_flag_0.gif?h=2ad7dc73&amp;itok=6LgLlzR9" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Iraq
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">Arabic</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Iraq%20NDC%20Document.docx" class="ndc-acr-download-link is-original" hreflang="ar">Iraq First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">Arabic</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Iraq%20NDC%20Document.docx" class="ndc-acr-download-link is-original" hreflang="ar">Iraq First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">15/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Iraq%20NDC%20Document.docx" class="ndc-acr-download-link is-original" hreflang="ar">Iraq First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497817 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nru_flag.gif?h=da9490b2&amp;itok=VsITSzig" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Nauru
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Nauru%20Updated%20NDC%20pdf.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Nauru%20Updated%20NDC%20pdf.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">14/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Nauru%20Updated%20NDC%20pdf.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497764 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/Flag_of_Mauritania.png?h=1c7500b8&amp;itok=yQvcTVsG" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mauritania
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN-actualis%C3%A9%202021_%20Mauritania.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mauritania First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN-actualis%C3%A9%202021_%20Mauritania.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mauritania First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN-actualis%C3%A9%202021_%20Mauritania.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mauritania First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497522 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gnb_flag.gif?h=27ba19da&amp;itok=CToFxVeU" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Guinea-Bissau
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC-Guinea%20Bissau-12102021.Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Guinea First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC-Guinea%20Bissau-12102021.Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Guinea First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC-Guinea%20Bissau-12102021.Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Guinea First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497650 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kwt_flag.gif?h=da9490b2&amp;itok=pZ7fmbb5" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kuwait
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kuwait%20updating%20the%20first%20NDC-English.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kuwait First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-original">Arabic</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kuwait%20updating%20the%20first%20NDC-arabic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Kuwait First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Arabic</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kuwait%20updating%20the%20first%20NDC-English.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kuwait First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kuwait%20updating%20the%20first%20NDC-arabic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Kuwait First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kuwait%20updating%20the%20first%20NDC-English.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kuwait First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kuwait%20updating%20the%20first%20NDC-arabic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Kuwait First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497605 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/jor_flag.gif?h=da9490b2&amp;itok=kZzHDVDC" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Jordan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/UPDATED%20SUBMISSION%20OF%20JORDANS.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jordan First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/UPDATED%20SUBMISSION%20OF%20JORDANS.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jordan First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/UPDATED%20SUBMISSION%20OF%20JORDANS.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jordan First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497963 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/swz_flag.gif?h=57cf074e&amp;itok=mU4Kq2Vd" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Eswatini
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Southern Africa" data-region-id="5899"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Eswatini%27s%20Revised%20NDC%2012%20Oct%202021.docx" class="ndc-acr-download-link is-original" hreflang="en">Eswatini First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Eswatini%27s%20Revised%20NDC%2012%20Oct%202021.docx" class="ndc-acr-download-link is-original" hreflang="en">Eswatini First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Eswatini%27s%20Revised%20NDC%2012%20Oct%202021.docx" class="ndc-acr-download-link is-original" hreflang="en">Eswatini First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497222 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/alb_flag.gif?h=57cf074e&amp;itok=-Uzdp_pC" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Albania
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2022-08/Albania%20Revised%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Albania First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2022-08/Albania%20Revised%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Albania First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2022-08/Albania%20Revised%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Albania First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497973 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tgo_flag.gif?h=27ba19da&amp;itok=EMdw5YTD" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Togo
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9es_Togo_Document%20int%C3%A9rimaire_rv_11%2010%2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Togo First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9es_Togo_Document%20int%C3%A9rimaire_rv_11%2010%2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Togo First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20Revis%C3%A9es_Togo_Document%20int%C3%A9rimaire_rv_11%2010%2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Togo First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497988 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tjk_flag.gif?h=da9490b2&amp;itok=aSXU4gqq" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Tajikistan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_TAJIKISTAN_ENG.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Tajikistan First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_TAJIKISTAN_RUSS.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Tajikistan First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Russian</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_TAJIKISTAN_ENG.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Tajikistan First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_TAJIKISTAN_RUSS.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Tajikistan First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_Letter_TJK.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tajikistan First NDC (Updated submission letter)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_TAJIKISTAN_ENG.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Tajikistan First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_TAJIKISTAN_RUSS.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Tajikistan First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_Letter_TJK.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tajikistan First NDC (Updated submission letter)</a></div>
</td>
</tr>
<tr class="submission-nid-497267 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ben_flag.gif?h=57cf074e&amp;itok=f4VIAgnz" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Benin
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_ACTUALISEE_BENIN2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Benin First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_ACTUALISEE_BENIN2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Benin First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_ACTUALISEE_BENIN2021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Benin First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497343 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/blr_flag_0.gif?h=da9490b2&amp;itok=qg9XIK2J" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Belarus
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belarus_NDC_English.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Belarus First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belarus_NDC_Russian.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Belarus First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Russian</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belarus_NDC_English.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Belarus First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belarus_NDC_Russian.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Belarus First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belarus_Letter.pdf" class="ndc-acr-download-link is-original" hreflang="en">Belarus First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">11/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belarus_NDC_English.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Belarus First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belarus_NDC_Russian.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Belarus First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belarus_Letter.pdf" class="ndc-acr-download-link is-original" hreflang="en">Belarus First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497406 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cmr_flag.gif?h=57cf074e&amp;itok=zxrBDFMA" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cameroon
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20r%C3%A9vis%C3%A9e%20CMR%20finale%20sept%202021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Cameroon First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">French</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20r%C3%A9vis%C3%A9e%20CMR%20finale%20sept%202021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Cameroon First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Lettre%20officielle%20CDN%20Cameroun%202021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Cameroon First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">11/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20r%C3%A9vis%C3%A9e%20CMR%20finale%20sept%202021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Cameroon First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Lettre%20officielle%20CDN%20Cameroun%202021.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Cameroon First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-499565 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mli_flag.gif?h=57cf074e&amp;itok=U49xp5Yz" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mali
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/MALI%20First%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mali First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/MALI%20First%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mali First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">11/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/MALI%20First%20NDC%20update.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Mali First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497887 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pse_flag.gif?h=fe61d327&amp;itok=KK1r_O5X" width="57" height="35" alt="" typeof="Image" class="img-responsive">
State of Palestine
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC_%20State%20of%20Palestine_2021_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">State of Palestine First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC_%20State%20of%20Palestine_2021_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">State of Palestine First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC_%20State%20of%20Palestine_2021_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">State of Palestine First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-498001 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tun_flag_0.gif?h=57cf074e&amp;itok=-57hHt6F" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Tunisia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Northern Africa" data-region-id="5896"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Tunisia%20Update%20NDC-french.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Tunisia First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-08/CDN%20-%20Updated%20-english%20version.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Updated NDC - English version</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Tunisia%20Update%20NDC-french.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Tunisia First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-08/CDN%20-%20Updated%20-english%20version.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Updated NDC - English version</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/CDN%20-%20updated%20executive%20summary.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated NDC - Executive summary, english version</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Tunisia%20Update%20NDC-french.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Tunisia First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-08/CDN%20-%20Updated%20-english%20version.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Updated NDC - English version</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-08/CDN%20-%20updated%20executive%20summary.pdf" class="ndc-acr-download-link is-original" hreflang="en">Updated NDC - Executive summary, english version</a></div>
</td>
</tr>
<tr class="submission-nid-497294 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bfa_flag.gif?h=57cf074e&amp;itok=u4hpEJnZ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Burkina Faso
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Rapport%20CDN_BKFA.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Rapport%20CDN_BKFA.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Rapport%20CDN_BKFA.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497631 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kgz_flag.gif?h=e57e762b&amp;itok=-hm6Wlad" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kyrgyzstan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%D0%9E%D0%9D%D0%A3%D0%92%20ENG%20%D0%BE%D1%82%2008102021.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kyrgyzstan First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%D0%9E%D0%9D%D0%A3%D0%92%20%D0%A0%D0%A3%D0%A1%20%D0%BE%D1%82%2008102021.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kyrgyzstan First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%D0%9E%D0%9D%D0%A3%D0%92%20ENG%20%D0%BE%D1%82%2008102021.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kyrgyzstan First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%D0%9E%D0%9D%D0%A3%D0%92%20%D0%A0%D0%A3%D0%A1%20%D0%BE%D1%82%2008102021.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kyrgyzstan First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%D0%9E%D0%9D%D0%A3%D0%92%20ENG%20%D0%BE%D1%82%2008102021.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kyrgyzstan First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%D0%9E%D0%9D%D0%A3%D0%92%20%D0%A0%D0%A3%D0%A1%20%D0%BE%D1%82%2008102021.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kyrgyzstan First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497767 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mus_flag.gif?h=57cf074e&amp;itok=dENyD122" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mauritius
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Updated%20NDC%20for%20the%20Republic%20of%20Mauritius%2001%20October%202021.docx" class="ndc-acr-download-link is-original" hreflang="en">Mauritius First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Updated%20NDC%20for%20the%20Republic%20of%20Mauritius%2001%20October%202021.docx" class="ndc-acr-download-link is-original" hreflang="en">Mauritius First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Mauritius%20Party%20Note.docx" class="ndc-acr-download-link is-original" hreflang="en">Mauritius Party Note</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Updated%20NDC%20for%20the%20Republic%20of%20Mauritius%2001%20October%202021.docx" class="ndc-acr-download-link is-original" hreflang="en">Mauritius First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Mauritius%20Party%20Note.docx" class="ndc-acr-download-link is-original" hreflang="en">Mauritius Party Note</a></div>
</td>
</tr>
<tr class="submission-nid-497265 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bdi_flag.gif?h=57cf074e&amp;itok=M1gP3ooO" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Burundi
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20%20%20Burundi%20ANNEXE%201.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20%20%20Burundi%20ANNEXE%201.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/10/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20%20%20Burundi%20ANNEXE%201.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-498051 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/zaf_flag.gif?h=57cf074e&amp;itok=hmgwDH56" width="57" height="35" alt="" typeof="Image" class="img-responsive">
South Africa
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Southern Africa" data-region-id="5899"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/South%20Africa%20updated%20first%20NDC%20September%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Africa First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/South%20Africa%20updated%20first%20NDC%20September%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Africa First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/09/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/South%20Africa%20updated%20first%20NDC%20September%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Africa First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497674 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lka_flag.gif?h=da9490b2&amp;itok=eTx_IqFv" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sri Lanka
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Amendmend%20to%20the%20Updated%20Nationally%20Determined%20Contributions%20of%20Sri%20Lanka.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sri Lanka First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Amendmend%20to%20the%20Updated%20Nationally%20Determined%20Contributions%20of%20Sri%20Lanka.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sri Lanka First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/09/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Amendmend%20to%20the%20Updated%20Nationally%20Determined%20Contributions%20of%20Sri%20Lanka.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sri Lanka First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-498055 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/zwe_flag.gif?h=b8db4804&amp;itok=-0MzRI73" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Zimbabwe
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Zimbabwe%20Revised%20Nationally%20Determined%20Contribution%202021%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zimbabwe First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Zimbabwe%20Revised%20Nationally%20Determined%20Contribution%202021%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zimbabwe First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/09/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Zimbabwe%20Revised%20Nationally%20Determined%20Contribution%202021%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zimbabwe First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497937 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ssd_flag.gif?h=5ed59bc2&amp;itok=bhjDRJUt" width="57" height="35" alt="" typeof="Image" class="img-responsive">
South Sudan
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/South%20Sudan%27s%20Second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Sudan Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/South%20Sudan%27s%20Second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Sudan Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">21/09/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/South%20Sudan%27s%20Second%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">South Sudan Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497524 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gmb_flag.gif?h=57cf074e&amp;itok=u_JThLZj" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Gambia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20NDC%20of%20The%20Republic%20of%20The%20Gambia-16-12-2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Gambia Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20NDC%20of%20The%20Republic%20of%20The%20Gambia-16-12-2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Gambia Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/09/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20NDC%20of%20The%20Republic%20of%20The%20Gambia-16-12-2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Gambia Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-499557 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/atg_flag.gif?h=57cf074e&amp;itok=HAoQXB9X" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Antigua and Barbuda
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/ATG%20-%20UNFCCC%20NDC%20-%202021-09-02%20-%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Antigua and Barbuda First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/ATG%20-%20UNFCCC%20NDC%20-%202021-09-02%20-%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Antigua and Barbuda First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">02/09/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/ATG%20-%20UNFCCC%20NDC%20-%202021-09-02%20-%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Antigua and Barbuda First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497340 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/blz_flag.gif?h=57cf074e&amp;itok=b6a37X0i" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Belize
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belize%20Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Belize First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belize%20Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Belize First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">01/09/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Belize%20Updated%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Belize First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497299 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bgd_flag.gif?h=27ba19da&amp;itok=2NLSJc3i" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bangladesh
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_submission_20210826revised.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bangladesh First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_submission_20210826revised.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bangladesh First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/08/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_submission_20210826revised.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bangladesh First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497878 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/qat_flag.gif?h=75e47415&amp;itok=E8KOovya" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Qatar
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Qatar%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Qatar First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-original">Arabic</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Qatar%20NDC%20-%20Arabic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Qatar First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Arabic</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Qatar%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Qatar First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Qatar%20NDC%20-%20Arabic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Qatar First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/submit%20the%20Nationally%20Determined%20Contribution%20to%20Executive%20Secretary%20of%20UNFCC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission letter_Qatar First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/08/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Qatar%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Qatar First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Qatar%20NDC%20-%20Arabic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Qatar First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/submit%20the%20Nationally%20Determined%20Contribution%20to%20Executive%20Secretary%20of%20UNFCC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission letter_Qatar First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497656 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lbr_flag.gif?h=6c49f853&amp;itok=cFEXbMjA" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Liberia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Liberia%27s%20Updated%20NDC_RL_FINAL%20%28002%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liberia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Liberia%27s%20Updated%20NDC_RL_FINAL%20%28002%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liberia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">04/08/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Liberia%27s%20Updated%20NDC_RL_FINAL%20%28002%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liberia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497746 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mmr_flag.gif?h=27ba19da&amp;itok=M9gskXfu" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Myanmar
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Myanmar%20Updated%20%20NDC%20July%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Myanmar First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Myanmar%20Updated%20%20NDC%20July%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Myanmar First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">03/08/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Myanmar%20Updated%20%20NDC%20July%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Myanmar First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497411 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cog_flag_0.gif?h=27ba19da&amp;itok=krp2rQDV" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Congo
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_Congo.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_Congo.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Lettre%20de%20transmission.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Submission letter</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">02/08/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN_Congo.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Lettre%20de%20transmission.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Submission letter</a></div>
</td>
</tr>
<tr class="submission-nid-498016 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ukr_flag.gif?h=d1720097&amp;itok=Rjq1ZHq3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ukraine
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ukraine%20NDC_July%2031.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ukraine First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ukraine%20NDC_July%2031.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ukraine First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ukraine%20NDC_July%2031.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ukraine First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497897 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sle_flag.gif?h=57cf074e&amp;itok=Fmqmw1rH" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sierra Leone
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/210804%202125%20SL%20NDC%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/210804%202125%20SL%20NDC%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/210804%202125%20SL%20NDC%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497911 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/som_flag.gif?h=57cf074e&amp;itok=Pjehj8Kk" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Somalia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Updated%20NDC%20for%20Somalia%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Somalia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Updated%20NDC%20for%20Somalia%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Somalia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Updated%20NDC%20for%20Somalia%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Somalia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497774 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mwi_flag.gif?h=57cf074e&amp;itok=GCkd8KGj" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Malawi
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Malawi%20Updated%20NDC%20July%202021%20submitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malawi First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Malawi%20Updated%20NDC%20July%202021%20submitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malawi First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Malawi%20NDC_Policy%20Brief_30%20June%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malawi NDC Policy Brief</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Malawi%20Updated%20NDC%20July%202021%20submitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malawi First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Malawi%20NDC_Policy%20Brief_30%20June%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malawi NDC Policy Brief</a></div>
</td>
</tr>
<tr class="submission-nid-498007 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tza_flag.gif?h=d4b38aaf&amp;itok=bI2lUCDQ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
United Republic of Tanzania
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/TANZANIA_NDC_SUBMISSION_30%20JULY%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">United Republic of Tanzania First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/TANZANIA_NDC_SUBMISSION_30%20JULY%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">United Republic of Tanzania First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/TANZANIA_NDC_SUBMISSION_30%20JULY%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">United Republic of Tanzania First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497776 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mys_flag.gif?h=b650b131&amp;itok=yiTRQvdW" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Malaysia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Malaysia%20NDC%20Updated%20Submission%20to%20UNFCCC%20July%202021%20final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malaysia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Malaysia%20NDC%20Updated%20Submission%20to%20UNFCCC%20July%202021%20final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malaysia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Malaysia%20NDC%20Updated%20Submission%20to%20UNFCCC%20July%202021%20final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Malaysia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-498057 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/zmb_flag.gif?h=57cf074e&amp;itok=Uq9A4N5D" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Zambia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Zambia_Revised%20and%20Updated_NDC_2021_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Zambia_Revised%20and%20Updated_NDC_2021_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Final%20Zambia_Revised%20and%20Updated_NDC_2021_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497965 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/Seychelles%20flag.jpg?h=c1beacee&amp;itok=ZZfRDObb" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Seychelles
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Seychelles%20-%20NDC_Jul30th%202021%20_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Seychelles First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Seychelles%20-%20NDC_Jul30th%202021%20_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Seychelles First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Seychelles%20-%20NDC_Jul30th%202021%20_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Seychelles First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-498047 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/wsm_flag.gif?h=b650b131&amp;itok=5PKln8Kt" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Samoa
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Samoa%27s%20Second%20NDC%20for%20UNFCCC%20Submission.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa Second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Samoa%27s%20Second%20NDC%20for%20UNFCCC%20Submission.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Note%20Verbale_UNFCCC_Samoa%27s%202nd%20NDCs%20300721.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa Note Verbale </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Samoa%27s%20Second%20NDC%20for%20UNFCCC%20Submission.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Note%20Verbale_UNFCCC_Samoa%27s%202nd%20NDCs%20300721.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa Note Verbale </a></div>
</td>
</tr>
<tr class="submission-nid-497356 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/brb_flag.gif?h=bfb37f4a&amp;itok=_XIege06" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Barbados
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/2021%20Barbados%20NDC%20update%20-%2021%20July%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Barbados First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/2021%20Barbados%20NDC%20update%20-%2021%20July%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Barbados First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/2021%20Barbados%20NDC%20update%20-%2021%20July%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Barbados First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497945 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/stp_flag.gif?h=da9490b2&amp;itok=GwE_lpL9" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sao Tome and Principe
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated_NDC_STP_2021_EN_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sao Tome and Principe First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated_NDC_STP_2021_EN_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sao Tome and Principe First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated_NDC_STP_2021_EN_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sao Tome and Principe First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497593 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/isr_flag.gif?h=dc257391&amp;itok=wd8su9qQ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Israel
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20update%20as%20submitted%20to%20the%20UNFCCC.docx" class="ndc-acr-download-link is-original" hreflang="en">Israel First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20update%20as%20submitted%20to%20the%20UNFCCC.docx" class="ndc-acr-download-link is-original" hreflang="en">Israel First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20update%20as%20submitted%20to%20the%20UNFCCC.docx" class="ndc-acr-download-link is-original" hreflang="en">Israel First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497518 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gin_flag.gif?h=57cf074e&amp;itok=R4VpVcwp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Guinea
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20GUINEE%202021_REVISION_VF.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20GUINEE%202021_REVISION_VF.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">28/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDN%20GUINEE%202021_REVISION_VF.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497487 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/eth_flag.gif?h=da9490b2&amp;itok=uZITk-Yc" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ethiopia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ethiopia%27s%20updated%20NDC%20JULY%202021%20Submission_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ethiopia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ethiopia%27s%20updated%20NDC%20JULY%202021%20Submission_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ethiopia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">23/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Ethiopia%27s%20updated%20NDC%20JULY%202021%20Submission_.pdf" class="ndc-acr-download-link is-original" hreflang="en">Ethiopia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497899 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/slb_flag.gif?h=27ee3080&amp;itok=8BeGXdi3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Solomon Islands
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Melanesia" data-region-id="5917"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Report%202021%20Final%20Solomon%20Islands%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Solomon Islands First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Report%202021%20Final%20Solomon%20Islands%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Solomon Islands First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Report%202021%20Final%20Solomon%20Islands%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Solomon Islands First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497854 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pry_flag.gif?h=da9490b2&amp;itok=xSwda_X3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Paraguay
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Actualizaci%C3%B3n-NDC%20VF%20PAG.%20WEB_MADES%20Mayo%202022.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>

<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Carta%20Formal_Remisi%C3%B3n%20Oficial%20de%20la%20Actualizaci%C3%B3n%20de%20la%20NDC%20de%20la%20Rep%C3%BAblica%20del%20Paraguay%20al%202030.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay Formal letter on submission of updated NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Actualizaci%C3%B3n-NDC%20VF%20PAG.%20WEB_MADES%20Mayo%202022.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Carta%20Formal_Remisi%C3%B3n%20Oficial%20de%20la%20Actualizaci%C3%B3n%20de%20la%20NDC%20de%20la%20Rep%C3%BAblica%20del%20Paraguay%20al%202030.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay Formal letter on submission of updated NDC</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Actualizaci%C3%B3n-NDC%20VF%20PAG.%20WEB_MADES%20Mayo%202022.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497381 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/can_flag.gif?h=b8db4804&amp;itok=mpdXI6mS" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Canada
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Northern America" data-region-id="5903"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Canada%27s%20Enhanced%20NDC%20Submission1_FINAL%20EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">Canada First NDC (updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Canada%27s%20Enhanced%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Canada First NDC (updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Canada%27s%20Enhanced%20NDC%20Submission1_FINAL%20EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">Canada First NDC (updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Canada%27s%20Enhanced%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Canada First NDC (updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/07/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Canada%27s%20Enhanced%20NDC%20Submission1_FINAL%20EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">Canada First NDC (updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Canada%27s%20Enhanced%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Canada First NDC (updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497368 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/btn_flag.gif?h=57cf074e&amp;itok=3EUOB0qp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bhutan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20NDC%20Bhutan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan Second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20NDC%20Bhutan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Forward%20Note.PDF" class="ndc-acr-download-link is-original" hreflang="en">Forward Note for submission of Second NDC </a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/06/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20NDC%20Bhutan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Forward%20Note.PDF" class="ndc-acr-download-link is-original" hreflang="en">Forward Note for submission of Second NDC </a></div>
</td>
</tr>
<tr class="submission-nid-497686 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mar_flag.gif?h=57cf074e&amp;itok=KZ58SgW7" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Morocco
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Northern Africa" data-region-id="5896"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Moroccan%20updated%20NDC%202021%20_Fr.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Morocco First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Moroccan%20updated%20NDC%202021%20_Fr.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Morocco First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/06/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Moroccan%20updated%20NDC%202021%20_Fr.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Morocco First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497751 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mne_flag.gif?h=d72776d6&amp;itok=NcrBvS3f" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Montenegro
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC%20for%20Montenegro.pdf" class="ndc-acr-download-link is-original" hreflang="en">Montenegro First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC%20for%20Montenegro.pdf" class="ndc-acr-download-link is-original" hreflang="en">Montenegro First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">15/06/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC%20for%20Montenegro.pdf" class="ndc-acr-download-link is-original" hreflang="en">Montenegro First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497220 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ago_flag.gif?h=c5aaa1f1&amp;itok=PKHSWfsT" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Angola
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Angola.pdf" class="ndc-acr-download-link is-original" hreflang="en">Angola First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Angola.pdf" class="ndc-acr-download-link is-original" hreflang="en">Angola First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/05/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Angola.pdf" class="ndc-acr-download-link is-original" hreflang="en">Angola First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-499560 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hnd_flag.gif?h=d4b38aaf&amp;itok=R8mDwun-" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Honduras
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20de%20Honduras_%20Primera%20Actualizaci%C3%B3n.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20de%20Honduras_%20Primera%20Actualizaci%C3%B3n.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Carta%20a%20la%20Sra.%20Patricia%20Espinosa%20%20Secretaria%20Ejecutiva%20CMNUCC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Presentacion de la Actualizacion de la Contribucion Nacional Determinada en su Primera Actualizacion</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/05/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20de%20Honduras_%20Primera%20Actualizaci%C3%B3n.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Carta%20a%20la%20Sra.%20Patricia%20Espinosa%20%20Secretaria%20Ejecutiva%20CMNUCC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Presentacion de la Actualizacion de la Contribucion Nacional Determinada en su Primera Actualizacion</a></div>
</td>
</tr>
<tr class="submission-nid-497648 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lao_flag.gif?h=2ad7dc73&amp;itok=iJQa7_aP" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Lao People's Democratic Republic
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%202020%20of%20Lao%20PDR%20%28English%29%2C%2009%20April%202021%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lao People's Democratic Republic First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%202020%20of%20Lao%20PDR%20%28English%29%2C%2009%20April%202021%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lao People's Democratic Republic First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">11/05/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%202020%20of%20Lao%20PDR%20%28English%29%2C%2009%20April%202021%20%281%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lao People's Democratic Republic First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497239 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/arm_flag.gif?h=57cf074e&amp;itok=3Ug20-D3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Armenia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20of%20Republic%20of%20Armenia%20%202021-2030.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20of%20Republic%20of%20Armenia%20%202021-2030.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Letter%20of%20the%20Minister%20of%20Environment.pdf" class="ndc-acr-download-link is-original" hreflang="en">Letter of the Minister of Environment in relation to the updated first NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/05/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20of%20Republic%20of%20Armenia%20%202021-2030.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Letter%20of%20the%20Minister%20of%20Environment.pdf" class="ndc-acr-download-link is-original" hreflang="en">Letter of the Minister of Environment in relation to the updated first NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497506 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/geo_flag.gif?h=57cf074e&amp;itok=PZlHoOLp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Georgia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Georgia_ENG%20WEB-approved.pdf" class="ndc-acr-download-link is-original" hreflang="en">Georgia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Georgia_ENG%20WEB-approved.pdf" class="ndc-acr-download-link is-original" hreflang="en">Georgia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/05/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20Georgia_ENG%20WEB-approved.pdf" class="ndc-acr-download-link is-original" hreflang="en">Georgia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-498021 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/usa_flag.gif?h=27ee3080&amp;itok=1dBodVbS" width="57" height="35" alt="" typeof="Image" class="img-responsive">
United States of America
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Northern America" data-region-id="5903"></span>(*) </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/United%20States%20NDC%20April%2021%202021%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">United States of America First NDC (After rejoining the Paris Agreement)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 167px;"><span class="field--name-field-set-item-language is-original" style="height: 167px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/United%20States%20NDC%20April%2021%202021%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">United States of America First NDC (After rejoining the Paris Agreement)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/04/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/United%20States%20NDC%20April%2021%202021%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">United States of America First NDC (After rejoining the Paris Agreement)</a></div>
</td>
</tr>
<tr class="submission-nid-497334 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bih_flag.gif?h=ade2d42a&amp;itok=MTWcw3FL" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bosnia and Herzegovina
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20BiH_November%202020%20FINAL%20DRAFT%2005%20Nov%20ENG%20LR.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bosnia and Herzegovina First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20BiH_November%202020%20FINAL%20DRAFT%2005%20Nov%20ENG%20LR.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bosnia and Herzegovina First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">20/04/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20BiH_November%202020%20FINAL%20DRAFT%2005%20Nov%20ENG%20LR.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bosnia and Herzegovina First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497727 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mkd_flag.gif?h=da9490b2&amp;itok=9w-NDmEE" width="57" height="35" alt="" typeof="Image" class="img-responsive">
North Macedonia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Macedonian%20enhanced%20NDC%20%28002%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of North Macedonia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Macedonian%20enhanced%20NDC%20%28002%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of North Macedonia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/04/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Macedonian%20enhanced%20NDC%20%28002%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of North Macedonia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497829 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/phl_flag.gif?h=b650b131&amp;itok=BY3Yatsx" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Philippines
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Philippines%20-%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Philippines First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Philippines%20-%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Philippines First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">15/04/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Philippines%20-%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Philippines First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497421 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cpv_flag.gif?h=e57e762b&amp;itok=OiYxshzb" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cabo Verde
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cabo%20Verde_NDC%20Update%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cabo Verde First NDC(Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cabo%20Verde_NDC%20Update%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cabo Verde First NDC(Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">02/04/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cabo%20Verde_NDC%20Update%202021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cabo Verde First NDC(Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497652 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lbn_flag.gif?h=2ad7dc73&amp;itok=MurLgFTB" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Lebanon
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Lebanon%27s%202020%20Nationally%20Determined%20Contribution%20Update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lebanon First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Lebanon%27s%202020%20Nationally%20Determined%20Contribution%20Update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lebanon First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/03/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Lebanon%27s%202020%20Nationally%20Determined%20Contribution%20Update.pdf" class="ndc-acr-download-link is-original" hreflang="en">Lebanon First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497591 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/isl_flag.gif?h=a745b187&amp;itok=6D6YcEH1" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Iceland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Northern Europe" data-region-id="5912"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Iceland_updated_NDC_Submission_Feb_2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Iceland_updated_NDC_Submission_Feb_2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">18/02/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Iceland_updated_NDC_Submission_Feb_2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Iceland First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497658 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lca_flag.gif?h=da9490b2&amp;itok=NyOJfrvn" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Saint Lucia
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Saint%20Lucia%20First%20NDC%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Lucia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Saint%20Lucia%20First%20NDC%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Lucia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/01/2021 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Saint%20Lucia%20First%20NDC%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Lucia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497352 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/brn_flag_0.gif?h=da9490b2&amp;itok=0N0fzB-l" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Brunei Darussalam
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="South-eastern Asia" data-region-id="5907"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Brunei%20Darussalam%27s%20NDC%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brunei Darussalam First NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Brunei%20Darussalam%27s%20NDC%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brunei Darussalam First NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%5B%2820%29%20KPN.CC.UN.1.30.12.2020%5D%20TO%20UNFCCC%20SECRETARIAT%20-%20Brunei%20Darussalam%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">[(20) KPN.CC.UN.1.30.12.2020] TO UNFCCC SECRETARIAT - Brunei Darussalam Nationally Determined Contribution</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Brunei%20Darussalam%27s%20NDC%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Brunei Darussalam First NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/%5B%2820%29%20KPN.CC.UN.1.30.12.2020%5D%20TO%20UNFCCC%20SECRETARIAT%20-%20Brunei%20Darussalam%20Nationally%20Determined%20Contribution.pdf" class="ndc-acr-download-link is-original" hreflang="en">[(20) KPN.CC.UN.1.30.12.2020] TO UNFCCC SECRETARIAT - Brunei Darussalam Nationally Determined Contribution</a></div>
</td>
</tr>
<tr class="submission-nid-497490 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fji_flag.gif?h=da9490b2&amp;itok=x6GQCrGd" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Fiji
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Melanesia" data-region-id="5917"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Republic%20of%20Fiji%27s%20Updated%20NDC%2020201.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Republic%20of%20Fiji%27s%20Updated%20NDC%2020201.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Republic%20of%20Fiji%27s%20Updated%20NDC%2020201.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497750 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mhl_flag.gif?h=27ee3080&amp;itok=dv-Jy3ab" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Marshall Islands
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/RMI%20NDC-UpdateUPDATED_01.20.2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Marshall Islands Second NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/RMI%20NDC-UpdateUPDATED_01.20.2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Marshall Islands Second NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/resource/180924%20rmi%202050%20climate%20strategy%20final.pdf" class="ndc-acr-download-link is-original" hreflang="en">180924 rmi 2050 climate strategy final_0</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/RMI%20Electricity%20Roadmap.pdf" class="ndc-acr-download-link is-original" hreflang="en">RMI Electricity Roadmap</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/RMI%20NDC-UpdateUPDATED_01.20.2021.pdf" class="ndc-acr-download-link is-original" hreflang="en">Marshall Islands Second NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/resource/180924%20rmi%202050%20climate%20strategy%20final.pdf" class="ndc-acr-download-link is-original" hreflang="en">180924 rmi 2050 climate strategy final_0</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/RMI%20Electricity%20Roadmap.pdf" class="ndc-acr-download-link is-original" hreflang="en">RMI Electricity Roadmap</a></div>
</td>
</tr>
<tr class="submission-nid-497432 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/col_flag.gif?h=57cf074e&amp;itok=eqy94Nxd" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Colombia
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 146px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Adjunto%202.%20%20Medidas%20de%20mitigaci%C3%B3n_NDC%20de%20Colombia%202020.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC-Technical Annex (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 146px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Adjunto%201.%20Metas%20de%20adaptaci%C3%B3n_NDC%20de%20Colombia%202020.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC-Technical Annex (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20actualizada%20de%20Colombia.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">Spanish</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">Spanish</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Adjunto%202.%20%20Medidas%20de%20mitigaci%C3%B3n_NDC%20de%20Colombia%202020.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC-Technical Annex (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Adjunto%201.%20Metas%20de%20adaptaci%C3%B3n_NDC%20de%20Colombia%202020.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC-Technical Annex (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20actualizada%20de%20Colombia.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/20.12.30%20S-GAA-20-027081%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="es">20.12.30 S-GAA-20-027081 NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Adjunto%202.%20%20Medidas%20de%20mitigaci%C3%B3n_NDC%20de%20Colombia%202020.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC-Technical Annex (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Adjunto%201.%20Metas%20de%20adaptaci%C3%B3n_NDC%20de%20Colombia%202020.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC-Technical Annex (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC%20actualizada%20de%20Colombia.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/20.12.30%20S-GAA-20-027081%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="es">20.12.30 S-GAA-20-027081 NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497468 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/Dominican%20Republic%20flag.jpg?h=e16fdc7a&amp;itok=UlZLdwpt" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Dominican Republic
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Dominican%20Republic%20First%20NDC%20%28Updated%20Submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Dominican Republic First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Dominican%20Republic%20First%20NDC%20%28Updated%20Submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Dominican Republic First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Dominican%20Republic%20First%20NDC%20%28Updated%20Submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Dominican Republic First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497450 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cri_flag.gif?h=27ba19da&amp;itok=iUi-adHk" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Costa Rica
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Contribucio%CC%81n%20Nacionalmente%20Determinada%20de%20Costa%20Rica%202020%20-%20Versio%CC%81n%20Completa.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Contribucio%CC%81n%20Nacionalmente%20Determinada%20de%20Costa%20Rica%202020%20-%20Versio%CC%81n%20Completa.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Contribucio%CC%81n%20Nacionalmente%20Determinada%20de%20Costa%20Rica%202020%20-%20Versio%CC%81n%20Completa.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497882 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sen_flag.gif?h=57cf074e&amp;itok=0PtHEya2" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Senegal
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDNSenegal%20approuv%C3%A9e-pdf-.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Senegal First NDC </a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDNSenegal%20approuv%C3%A9e-pdf-.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Senegal First NDC </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/lettre%20de%20trans%20Rapport%20CDN%20S%C3%A9n%C3%A9gal.pdf" class="ndc-acr-download-link is-original" hreflang="fr">lettre de trans Rapport CDN Sénégal</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CDNSenegal%20approuv%C3%A9e-pdf-.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Senegal First NDC </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/lettre%20de%20trans%20Rapport%20CDN%20S%C3%A9n%C3%A9gal.pdf" class="ndc-acr-download-link is-original" hreflang="fr">lettre de trans Rapport CDN Sénégal</a></div>
</td>
</tr>
<tr class="submission-nid-497613 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ken_flag.gif?h=57cf074e&amp;itok=DKa8bTUE" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kenya
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kenya%27s%20First%20%20NDC%20%28updated%20version%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kenya First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kenya%27s%20First%20%20NDC%20%28updated%20version%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kenya First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">28/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Kenya%27s%20First%20%20NDC%20%28updated%20version%29.pdf" class="ndc-acr-download-link is-original" hreflang="en">Kenya First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497699 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/120px-Maldives_flag_300.png?h=bfb37f4a&amp;itok=ZX15KCRc" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Maldives
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Maldives%20Nationally%20Determined%20Contribution%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Maldives First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Maldives%20Nationally%20Determined%20Contribution%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Maldives First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">28/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Maldives%20Nationally%20Determined%20Contribution%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Maldives First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497694 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mco_flag.gif?h=d4b38aaf&amp;itok=EgWt5YfP" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Monaco
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Monaco_NDC_2020.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Monaco First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Monaco_NDC_2020.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Monaco First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">28/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Monaco_NDC_2020.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Monaco First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-499568 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nic_flag.gif?h=27ba19da&amp;itok=5OxBWN7b" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Nicaragua
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Contribuciones_Nacionales_Determinadas_Nicaragua.pdf" class="ndc-acr-download-link is-original" hreflang="es">Nicaragua First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Contribuciones_Nacionales_Determinadas_Nicaragua.pdf" class="ndc-acr-download-link is-original" hreflang="es">Nicaragua First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/ACTUALIZACI%C3%93N%20DE%20CONTRIBUCIONES.pdf" class="ndc-acr-download-link is-original" hreflang="es">ACTUALIZACIÓN DE CONTRIBUCIONES</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Carta%20Sra.%20Patricia%20Espinosa.pdf" class="ndc-acr-download-link is-original" hreflang="es">Carta Sra. Patricia Espinosa</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Contribuciones_Nacionales_Determinadas_Nicaragua.pdf" class="ndc-acr-download-link is-original" hreflang="es">Nicaragua First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/ACTUALIZACI%C3%93N%20DE%20CONTRIBUCIONES.pdf" class="ndc-acr-download-link is-original" hreflang="es">ACTUALIZACIÓN DE CONTRIBUCIONES</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Carta%20Sra.%20Patricia%20Espinosa.pdf" class="ndc-acr-download-link is-original" hreflang="es">Carta Sra. Patricia Espinosa</a></div>
</td>
</tr>
<tr class="submission-nid-499570 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/per_flag.gif?h=57cf074e&amp;itok=qorWi1PB" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Peru
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Reporte%20de%20Actualizacio%CC%81n%20de%20las%20NDC%20del%20Peru%CC%81.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Reporte%20de%20Actualizacio%CC%81n%20de%20las%20NDC%20del%20Peru%CC%81.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">18/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Reporte%20de%20Actualizacio%CC%81n%20de%20las%20NDC%20del%20Peru%CC%81.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497851 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/png_flag.gif?h=999d46c7&amp;itok=UvHdNVHF" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Papua New Guinea
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Melanesia" data-region-id="5917"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/PNG%20Enhanced%20NDC%202020%20Summary.pdf" class="ndc-acr-download-link is-original" hreflang="en">Papua New Guinea Second NDC Summary </a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/PNG%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Papua New Guinea Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/PNG%20Enhanced%20NDC%202020%20Summary.pdf" class="ndc-acr-download-link is-original" hreflang="en">Papua New Guinea Second NDC Summary </a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/PNG%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Papua New Guinea Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">16/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/PNG%20Enhanced%20NDC%202020%20Summary.pdf" class="ndc-acr-download-link is-original" hreflang="en">Papua New Guinea Second NDC Summary </a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/PNG%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Papua New Guinea Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497995 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ton_flag.gif?h=da9490b2&amp;itok=KH1tJU-R" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Tonga
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Tonga%27s%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tonga Second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Tonga%27s%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tonga Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Tonga%20NDC%20Review%20Report.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tonga Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Tonga%27s%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tonga Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Tonga%20NDC%20Review%20Report.pdf" class="ndc-acr-download-link is-original" hreflang="en">Tonga Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497813 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/npl_flag.gif?h=3ffbe2c7&amp;itok=LNckKNR-" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Nepal
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20Nationally%20Determined%20Contribution%20%28NDC%29%20-%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nepal Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20Nationally%20Determined%20Contribution%20%28NDC%29%20-%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nepal Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Second%20Nationally%20Determined%20Contribution%20%28NDC%29%20-%202020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nepal Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497530 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/grd_flag.gif?h=27ba19da&amp;itok=09fp-5cj" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Grenada
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/GrenadaSecondNDC2020%20-%2001-12-20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Greneda Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/GrenadaSecondNDC2020%20-%2001-12-20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Greneda Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">01/12/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/GrenadaSecondNDC2020%20-%2001-12-20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Greneda Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497863 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/rus_flag.gif?h=57cf074e&amp;itok=XREo3dah" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Russian Federation
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_RF_eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Russian Federation First NDC</a><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_RF_ru.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Russian Federation First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_RF_eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Russian Federation First NDC</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_RF_ru.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Russian Federation First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">25/11/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_RF_eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Russian Federation First NDC</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_RF_ru.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Russian Federation First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497754 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mng_flag.gif?h=da9490b2&amp;itok=hi3Nm8CI" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mongolia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/First%20Submission%20of%20Mongolia%27s%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mongolia First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/First%20Submission%20of%20Mongolia%27s%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mongolia First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/10/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/First%20Submission%20of%20Mongolia%27s%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mongolia First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497442 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cub_flag_0.gif?h=da9490b2&amp;itok=e9yt4jRh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cuba
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20Summary%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Cuba First NDC Summary (Updated submission)</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20%28Updated%20submission%291.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Cuba First NDC (Updated submission)</a><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Cuba First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20Summary%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Cuba First NDC Summary (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20%28Updated%20submission%291.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Cuba First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Cuba First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/09/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20Summary%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Cuba First NDC Summary (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20%28Updated%20submission%291.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Cuba First NDC (Updated submission)</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cuban%20First%20NDC%20%28Updated%20submission%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Cuba First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497601 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/jam_flag.gif?h=b8db4804&amp;itok=ZlnsCAiy" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Jamaica
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC%20Jamaica%20-%20ICTU%20Guidance.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jamaica First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC%20Jamaica%20-%20ICTU%20Guidance.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jamaica First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">01/07/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Updated%20NDC%20Jamaica%20-%20ICTU%20Guidance.pdf" class="ndc-acr-download-link is-original" hreflang="en">Jamaica First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497874 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/rwa_flag.gif?h=57cf074e&amp;itok=aCAcWVpz" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Rwanda
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Rwanda_Updated_NDC_May_2020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Rwanda_Updated_NDC_May_2020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Submission%20letter%20of%20Updated%20NDC%20for%20Rwanda.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission letter of Updated NDC for Rwanda</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">20/05/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Rwanda_Updated_NDC_May_2020.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Submission%20letter%20of%20Updated%20NDC%20for%20Rwanda.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission letter of Updated NDC for Rwanda</a></div>
</td>
</tr>
<tr class="submission-nid-499044 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/chl_flag.gif?h=57cf074e&amp;itok=zAAmLFG_" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Chile
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_Chile_2020_espan%CC%83ol.pdf" class="ndc-acr-download-link is-original" hreflang="es">Chile First NDC (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Chile%27s_NDC_2020_english.pdf" class="ndc-acr-download-link is-original" hreflang="en">Chile First NDC (Updated submission)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_Chile_2020_espan%CC%83ol.pdf" class="ndc-acr-download-link is-original" hreflang="es">Chile First NDC (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Chile%27s_NDC_2020_english.pdf" class="ndc-acr-download-link is-original" hreflang="en">Chile First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Chile_%20fortalecimiento%20NDC_nov22.pdf" class="ndc-acr-download-link is-original" hreflang="es">Chile's NDC Strengthening Annex</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/04/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NDC_Chile_2020_espan%CC%83ol.pdf" class="ndc-acr-download-link is-original" hreflang="es">Chile First NDC (Updated submission)</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Chile%27s_NDC_2020_english.pdf" class="ndc-acr-download-link is-original" hreflang="en">Chile First NDC (Updated submission)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-11/Chile_%20fortalecimiento%20NDC_nov22.pdf" class="ndc-acr-download-link is-original" hreflang="es">Chile's NDC Strengthening Annex</a></div>
</td>
</tr>
<tr class="submission-nid-497691 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mda_flag.gif?h=da9490b2&amp;itok=P38ufCeK" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Republic of Moldova
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/MD_Updated_NDC_final_version_EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Moldova First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/MD_Updated_NDC_final_version_EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Moldova First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">04/03/2020 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/MD_Updated_NDC_final_version_EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">Republic of Moldova First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497949 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sur_flag.gif?h=57cf074e&amp;itok=wWHdlhA1" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Suriname
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Suriname%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Suriname Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Suriname%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Suriname Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/12/2019 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Suriname%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Suriname Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497843 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/prk_flag.gif?h=da9490b2&amp;itok=Uc-YT81u" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Democratic People's Republic of Korea
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/2019.09.19_DPRK%20letter%20to%20SG%20special%20envoy%20for%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Democratic People's Republic of Korea First NDC (Updated submission)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/2019.09.19_DPRK%20letter%20to%20SG%20special%20envoy%20for%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Democratic People's Republic of Korea First NDC (Updated submission)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/09/2019 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/2019.09.19_DPRK%20letter%20to%20SG%20special%20envoy%20for%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Democratic People's Republic of Korea First NDC (Updated submission)</a></div>
</td>
</tr>
<tr class="submission-nid-497460 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ecu_flag.gif?h=da9490b2&amp;itok=TUBnN1Bu" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ecuador
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Primera%20NDC%20Ecuador.pdf" class="ndc-acr-download-link is-original" hreflang="es">Ecuador First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Primera%20NDC%20Ecuador.pdf" class="ndc-acr-download-link is-original" hreflang="es">Ecuador First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/03/2019 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Primera%20NDC%20Ecuador.pdf" class="ndc-acr-download-link is-original" hreflang="es">Ecuador First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497961 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/Flag_of_Syria_%282025%29.gif?h=1c7500b8&amp;itok=yvOSksvl" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Syrian Arab Republic
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/FirstNDC-Eng-Syrian%20Arab%20Republic.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Syrian Arab Republic First NDC</a><span class="field--name-field-set-item-language is-original">Arabic</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/First%20NDC_Syrian%20Arab%20Republic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Syrian Arab Republic First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Arabic</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/FirstNDC-Eng-Syrian%20Arab%20Republic.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Syrian Arab Republic First NDC</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/First%20NDC_Syrian%20Arab%20Republic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Syrian Arab Republic First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">30/11/2018 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/FirstNDC-Eng-Syrian%20Arab%20Republic.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Syrian Arab Republic First NDC</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/First%20NDC_Syrian%20Arab%20Republic.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Syrian Arab Republic First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497903 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/Flag_of_San_Marino.svg_.png?h=24899386&amp;itok=OrfNJTE7" width="57" height="35" alt="" typeof="Image" class="img-responsive">
San Marino
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Southern Europe" data-region-id="5913"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/SAN%20MARINO%20INDC%20EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">San Marino First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/SAN%20MARINO%20INDC%20EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">San Marino First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/09/2018 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/SAN%20MARINO%20INDC%20EN.pdf" class="ndc-acr-download-link is-original" hreflang="en">San Marino First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497552 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/eri_flag.gif?h=b650b131&amp;itok=4quwm5Mq" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Eritrea
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NRC%20Eritrea.pdf" class="ndc-acr-download-link is-original" hreflang="en">Eritrea First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NRC%20Eritrea.pdf" class="ndc-acr-download-link is-original" hreflang="en">Eritrea First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/06/2018 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/NRC%20Eritrea.pdf" class="ndc-acr-download-link is-original" hreflang="en">Eritrea First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497985 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tto_flag.gif?h=27ba19da&amp;itok=NAuBcAoy" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Trinidad and Tobago
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Trinidad%20and%20Tobago%20Final%20INDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Trinidad%20and%20Tobago%20Final%20INDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/02/2018 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Trinidad%20and%20Tobago%20Final%20INDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497646 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lie_flag.gif?h=27ba19da&amp;itok=B3CTc8dA" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Liechtenstein
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Western Europe" data-region-id="5914"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/150422_INDC_FL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liechtenstein First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/150422_INDC_FL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liechtenstein First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">20/09/2017 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/150422_INDC_FL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Liechtenstein First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497212 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/afg_flag.gif?h=c5aaa1f1&amp;itok=1VRWZCg_" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Afghanistan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/INDC_AFG_20150927_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Afghanistan First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/INDC_AFG_20150927_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Afghanistan First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">23/11/2016 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/INDC_AFG_20150927_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Afghanistan First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497361 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bwa_flag_0.gif?h=57cf074e&amp;itok=FRTB4C3o" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Botswana
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Southern Africa" data-region-id="5899"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/BOTSWANA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Botswana First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/BOTSWANA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Botswana First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">11/11/2016 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/BOTSWANA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Botswana First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497447 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/dji_flag.gif?h=f3674879&amp;itok=tRV1VqIr" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Djibouti
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/INDC-Djibouti_ENG.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Djibouti First NDC</a><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CPDN%20Djibouti_9%20-%20CPDN%20-%20Format%20pour%20soumission%20CCNUCC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/INDC-Djibouti_ENG.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Djibouti First NDC</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CPDN%20Djibouti_9%20-%20CPDN%20-%20Format%20pour%20soumission%20CCNUCC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">11/11/2016 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/INDC-Djibouti_ENG.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Djibouti First NDC</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/CPDN%20Djibouti_9%20-%20CPDN%20-%20Format%20pour%20soumission%20CCNUCC.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497456 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/dza_flag.gif?h=bfb37f4a&amp;itok=kTn6i6vo" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Algeria
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Northern Africa" data-region-id="5896"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Algeria%20-%20INDC%20%28English%20unofficial%20translation%29%20September%2003%2C2015.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Algeria First NDC Translation</a><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Alg%C3%A9rie%20-INDC-%2003%20septembre%202015.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Algeria First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Algeria%20-%20INDC%20%28English%20unofficial%20translation%29%20September%2003%2C2015.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Algeria First NDC Translation</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Alg%C3%A9rie%20-INDC-%2003%20septembre%202015.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Algeria First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">20/10/2016 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Algeria%20-%20INDC%20%28English%20unofficial%20translation%29%20September%2003%2C2015.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Algeria First NDC Translation</a><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Alg%C3%A9rie%20-INDC-%2003%20septembre%202015.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Algeria First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497403 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cok_flag.gif?h=da9490b2&amp;itok=HY0aHcwi" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cook Islands
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cook%20Islands%20INDCsFINAL7Nov.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cook Islands First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cook%20Islands%20INDCsFINAL7Nov.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cook Islands First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">01/09/2016 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Cook%20Islands%20INDCsFINAL7Nov.pdf" class="ndc-acr-download-link is-original" hreflang="en">Cook Islands First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-498022 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/vct_flag.gif?h=57cf074e&amp;itok=6Df69H8G" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Saint Vincent and the Grenadines
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Saint%20Vincent%20and%20the%20Grenadines_NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Vincent and the Grenadines First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Saint%20Vincent%20and%20the%20Grenadines_NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Vincent and the Grenadines First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/06/2016 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Saint%20Vincent%20and%20the%20Grenadines_NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saint Vincent and the Grenadines First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497559 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/guy_flag.gif?h=d4b38aaf&amp;itok=6YLNOEXw" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Guyana
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Guyana%27s%20revised%20NDC%20-%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Guyana First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Guyana%27s%20revised%20NDC%20-%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Guyana First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">20/05/2016 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Guyana%27s%20revised%20NDC%20-%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Guyana First NDC</a></div>
</td>
</tr>
<tr class="submission-nid-497832 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/plw_flag.gif?h=fe1b3b8f&amp;itok=fiPF38b3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Palau
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Palau_INDC.Final%20Copy.pdf" class="ndc-acr-download-link is-original" hreflang="en">Palau First NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Palau_INDC.Final%20Copy.pdf" class="ndc-acr-download-link is-original" hreflang="en">Palau First NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_1">1</span><span class="alt_0 ndc_submission_version_1">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">22/04/2016 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/NDC/2022-06/Palau_INDC.Final%20Copy.pdf" class=<div class="view-content">
<div class="table-responsive">
<table class="table table-hover table-striped">
<thead>
<tr class="processed">
<th id="view-title-table-column" class="views-field views-field-title" scope="col"><a href="?field_party_region_target_id=All&amp;field_document_ca_target_id=All&amp;field_vd_status_target_id=5933&amp;start_date_datepicker=&amp;end_date_datepicker=&amp;order=title&amp;sort=asc" title="sort by Party">Party</a></th>
<th id="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments" scope="col">Title</th>
<th id="view-nothing-table-column" class="views-field views-field-nothing" scope="col">Language</th>
<th id="view-nothing-1-table-column" class="views-field views-field-nothing-1" scope="col">Translation</th>
<th id="view-field-version-number-table-column" class="views-field views-field-field-version-number" scope="col"><a href="?field_party_region_target_id=All&amp;field_document_ca_target_id=All&amp;field_vd_status_target_id=5933&amp;start_date_datepicker=&amp;end_date_datepicker=&amp;order=field_version_number&amp;sort=desc" title="sort by Version">Version</a></th>
<th id="view-field-vd-status-table-column" class="views-field views-field-field-vd-status" scope="col">Status</th>
<th id="view-field-document-sb-table-column" aria-sort="descending" class="views-field views-field-field-document-sb is-active" scope="col"><a href="?field_party_region_target_id=All&amp;field_document_ca_target_id=All&amp;field_vd_status_target_id=5933&amp;start_date_datepicker=&amp;end_date_datepicker=&amp;order=field_document_sb&amp;sort=asc" title="sort by Submission Date">Submission Date<span class="icon glyphicon glyphicon-chevron-down icon-after" aria-hidden="true" data-toggle="tooltip" data-placement="bottom" title="" data-original-title="Sort ascending"></span>
</a></th>
<th id="view-nothing-2-table-column" class="views-field views-field-nothing-2" scope="col">Additional documents</th>
</tr>
</thead>
<tbody>
<tr class="submission-nid-655674 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cog_flag_0.gif?h=27ba19da&amp;itok=krp2rQDV" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Congo
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2026-02/CDN%203.0%20de%20la%20R%C3%A9publique%20du%20Congo%20version%20finale.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-02/CDN%203.0%20de%20la%20R%C3%A9publique%20du%20Congo%20version%20finale.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-02/M.%20Simon%20Stiell%20_Secre%CC%81taire%20Exe%CC%81cutif%20de%20la%20Convention%20Cadre%20des%20Nations%20Unies%20sur%20le%20Changement%20Climatique%20_Ts%20de%20la%20CDN%203.0%20en%20Re%CC%81publique%20du%20Congo.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Submission letter</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/02/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-02/CDN%203.0%20de%20la%20R%C3%A9publique%20du%20Congo%20version%20finale.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Congo NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-02/M.%20Simon%20Stiell%20_Secre%CC%81taire%20Exe%CC%81cutif%20de%20la%20Convention%20Cadre%20des%20Nations%20Unies%20sur%20le%20Changement%20Climatique%20_Ts%20de%20la%20CDN%203.0%20en%20Re%CC%81publique%20du%20Congo.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Submission letter</a></div>
</td>
</tr>
<tr class="submission-nid-655523 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/plw_flag.gif?h=fe1b3b8f&amp;itok=fiPF38b3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Palau
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Palau%20Final%20Endorsed%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Palau NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Palau%20Final%20Endorsed%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Palau NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">29/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Palau%20Final%20Endorsed%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Palau NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655496 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tto_flag.gif?h=27ba19da&amp;itok=NAuBcAoy" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Trinidad and Tobago
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Trinidad%20and%20Tobago%20Second%20NDC%20%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago Second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Trinidad%20and%20Tobago%20Second%20NDC%20%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-01/Letter%20to%20the%20UNFCCC%20-%20Submission%20of%20T%26T%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Letter of submission Trinidad and Tobago</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_2">2</span><span class="alt_0 ndc_submission_version_2">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Trinidad%20and%20Tobago%20Second%20NDC%20%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Trinidad and Tobago Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2026-01/Letter%20to%20the%20UNFCCC%20-%20Submission%20of%20T%26T%20Second%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">Letter of submission Trinidad and Tobago</a></div>
</td>
</tr>
<tr class="submission-nid-655445 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hnd_flag.gif?h=d4b38aaf&amp;itok=R8mDwun-" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Honduras
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2026-01/HON%20NDC%203.0%202026%20Oficial.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras NDC 3.0 </a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/HON%20NDC%203.0%202026%20Oficial.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">21/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/HON%20NDC%203.0%202026%20Oficial.pdf" class="ndc-acr-download-link is-original" hreflang="es">Honduras NDC 3.0 </a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655394 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/wsm_flag.gif?h=b650b131&amp;itok=5PKln8Kt" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Samoa
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Polynesia" data-region-id="5919"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Samoa%20NDC3.0_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Samoa%20NDC3.0_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">14/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Samoa%20NDC3.0_FINAL.pdf" class="ndc-acr-download-link is-original" hreflang="en">Samoa NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655361 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/nru_flag.gif?h=da9490b2&amp;itok=VsITSzig" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Nauru
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Micronesia" data-region-id="5918"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Nauru%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Nauru%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/01/2026 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Nauru%20NDC%203.pdf" class="ndc-acr-download-link is-original" hreflang="en">Nauru NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655391 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/arm_flag.gif?h=57cf074e&amp;itok=3Ug20-D3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Armenia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/Armenia%27s%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Armenia%27s%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/Armenia%27s%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Armenia NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655360 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bfa_flag.gif?h=57cf074e&amp;itok=u4hpEJnZ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Burkina Faso
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2026-01/BKF-CDN%203.0_BURKINA%20FASO.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/BKF-CDN%203.0_BURKINA%20FASO.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/BKF-CDN%203.0_BURKINA%20FASO.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burkina Faso NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655320 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sau_flag.gif?h=57cf074e&amp;itok=ZaKxwcmX" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Saudi Arabia
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-01/2nd%20KSA%20NDC%20Document%20Final%20Document%20%2828122025%29_DNA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia Second NDC</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/2nd%20KSA%20NDC%20Document%20Final%20Document%20%2828122025%29_DNA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">31/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-01/2nd%20KSA%20NDC%20Document%20Final%20Document%20%2828122025%29_DNA.pdf" class="ndc-acr-download-link is-original" hreflang="en">Saudi Arabia Second NDC</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655301 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kor_flag.gif?h=57cf074e&amp;itok=TMP5hws2" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Republic of Korea
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Eastern Asia" data-region-id="5906"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-12/The%20Republic%20of%20Koreas%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of Korea's 2035 NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/The%20Republic%20of%20Koreas%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of Korea's 2035 NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/The%20Republic%20of%20Koreas%202035%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The Republic of Korea's 2035 NDC</a></div>
</td>
</tr>
<tr class="submission-nid-655299 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/slv_flag.gif?h=f3674879&amp;itok=GZlrX9HT" width="57" height="35" alt="" typeof="Image" class="img-responsive">
El Salvador
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-12/NDC%20EL%20SALVADOR%202025-%20VF.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC%20EL%20SALVADOR%202025-%20VF.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC%20EL%20SALVADOR%202025-%20VF.pdf" class="ndc-acr-download-link is-original" hreflang="es">El Salvador NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655298 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gab_flag.gif?h=d4b38aaf&amp;itok=sL8Y0n4I" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Gabon
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Middle Africa" data-region-id="5898"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-12/NDC3.0_Gabon_5_11_2025-final%20version.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Gabon NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC3.0_Gabon_5_11_2025-final%20version.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Gabon NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">24/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC3.0_Gabon_5_11_2025-final%20version.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Gabon NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655274 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/col_flag.gif?h=57cf074e&amp;itok=eqy94Nxd" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Colombia
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-12/NDC%203.0%20Transformaciones%20para%20la%20Vida%20%28Colombia%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC%203.0%20Transformaciones%20para%20la%20Vida%20%28Colombia%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">19/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/NDC%203.0%20Transformaciones%20para%20la%20Vida%20%28Colombia%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Colombia NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655679 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/zmb_flag.gif?h=57cf074e&amp;itok=Uq9A4N5D" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Zambia
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2026-02/1Final%20Submission%20of%20Zambia%20NDC%203.0%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-02/1Final%20Submission%20of%20Zambia%20NDC%203.0%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">15/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2026-02/1Final%20Submission%20of%20Zambia%20NDC%203.0%20.pdf" class="ndc-acr-download-link is-original" hreflang="en">Zambia NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655182 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/sle_flag.gif?h=57cf074e&amp;itok=Fmqmw1rH" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Sierra Leone
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-12/Sierra%20Leone%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/Sierra%20Leone%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/Sierra%20Leone%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Sierra Leone NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655178 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/rwa_flag.gif?h=57cf074e&amp;itok=aCAcWVpz" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Rwanda
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-12/Rwanda%20NDC3.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/Rwanda%20NDC3.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">08/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/Rwanda%20NDC3.0%20Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Rwanda NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-655117 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bhr_flag.gif?h=d4b38aaf&amp;itok=cJULjQL0" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bahrain
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-12/30112025_Bahrain_2025NDC3.0_vSubmitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/30112025_Bahrain_2025NDC3.0_vSubmitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">01/12/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-12/30112025_Bahrain_2025NDC3.0_vSubmitted.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahrain NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655104 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pan_flag.gif?h=57cf074e&amp;itok=QSPIis5C" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Panama
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/Pacto%20de%20Panam%C3%A1%20con%20la%20Naturaleza%20%28Nature%20Pledge%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Pacto%20de%20Panam%C3%A1%20con%20la%20Naturaleza%20%28Nature%20Pledge%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">27/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Pacto%20de%20Panam%C3%A1%20con%20la%20Naturaleza%20%28Nature%20Pledge%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Panama NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-655101 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/kaz_flag.gif?h=da9490b2&amp;itok=6gGhFnLE" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Kazakhstan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC_Kazakhstan%203.0%20eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kazakhstan NDC 3.0</a><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC%20Kazakhstan%203.0%20russ.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kazakhstan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC_Kazakhstan%203.0%20eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kazakhstan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-11/NDC%20Kazakhstan%203.0%20russ.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kazakhstan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">26/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC_Kazakhstan%203.0%20eng.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Kazakhstan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-11/NDC%20Kazakhstan%203.0%20russ.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Kazakhstan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-654987 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/qat_flag.gif?h=75e47415&amp;itok=E8KOovya" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Qatar
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Qatar%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Qatar NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Qatar%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Qatar NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">21/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Qatar%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Qatar NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-654345 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/mex_flag.gif?h=27ba19da&amp;itok=ah4BCLpQ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mexico
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Me%CC%81xico_spanish.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Me%CC%81xico_spanish.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Me%CC%81xico_spanish.pdf" class="ndc-acr-download-link is-original" hreflang="es">Mexico NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-654232 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/yem_flag_0.gif?h=57cf074e&amp;itok=SlwPslYs" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Yemen
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span>(*) </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Yemens%20NDC3.0%20Vision.pdf" class="ndc-acr-download-link is-original" hreflang="en">Yemen NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Yemens%20NDC3.0%20Vision.pdf" class="ndc-acr-download-link is-original" hreflang="en">Yemen NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">17/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Yemens%20NDC3.0%20Vision.pdf" class="ndc-acr-download-link is-original" hreflang="en">Yemen NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-654085 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cri_flag.gif?h=27ba19da&amp;itok=iUi-adHk" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Costa Rica
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/CND-2025-2035.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica NDC 2025 - 2035</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CND-2025-2035.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica NDC 2025 - 2035</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">14/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CND-2025-2035.pdf" class="ndc-acr-download-link is-original" hreflang="es">Costa Rica NDC 2025 - 2035</a></div>
</td>
</tr>
<tr class="submission-nid-653770 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/irq_flag_0.gif?h=2ad7dc73&amp;itok=6LgLlzR9" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Iraq
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">Arabic</span><a href="https://unfccc.int/sites/default/files/2025-11/%D9%88%D8%AB%D9%8A%D9%82%D8%A9%20%D8%A7%D9%84%D9%85%D8%B3%D8%A7%D9%87%D9%85%D8%A7%D8%AA%20%D8%A7%D9%84%D9%85%D8%AD%D8%AF%D8%AF%D8%A9%20%D9%88%D8%B7%D9%86%D9%8A%D8%A7%20-%202025.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Iraq NDC 3.0</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2026-01/NDC%20Report%20EN%20-%202025.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Iraq NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">Arabic</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/%D9%88%D8%AB%D9%8A%D9%82%D8%A9%20%D8%A7%D9%84%D9%85%D8%B3%D8%A7%D9%87%D9%85%D8%A7%D8%AA%20%D8%A7%D9%84%D9%85%D8%AD%D8%AF%D8%AF%D8%A9%20%D9%88%D8%B7%D9%86%D9%8A%D8%A7%20-%202025.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Iraq NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2026-01/NDC%20Report%20EN%20-%202025.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Iraq NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">13/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/%D9%88%D8%AB%D9%8A%D9%82%D8%A9%20%D8%A7%D9%84%D9%85%D8%B3%D8%A7%D9%87%D9%85%D8%A7%D8%AA%20%D8%A7%D9%84%D9%85%D8%AD%D8%AF%D8%AF%D8%A9%20%D9%88%D8%B7%D9%86%D9%8A%D8%A7%20-%202025.pdf" class="ndc-acr-download-link is-original" hreflang="ar">Iraq NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2026-01/NDC%20Report%20EN%20-%202025.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Iraq NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-653502 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/dji_flag.gif?h=f3674879&amp;itok=tRV1VqIr" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Djibouti
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-11/CDN%20revisee_Djibouti_Novembre%202025.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN%20revisee_Djibouti_Novembre%202025.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">12/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN%20revisee_Djibouti_Novembre%202025.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Djibouti Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-653298 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/ukr_flag.gif?h=d1720097&amp;itok=Rjq1ZHq3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Ukraine
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 42px;"><a href="https://unfccc.int/sites/default/files/2025-11/2%20%D0%9D%D0%92%D0%922%20%D0%BF%D1%80%D0%BE%D1%94%D0%BA%D1%82%20%28%D0%B7%D0%BC%D1%96%D0%BD%D0%B5%D0%BD%D0%B0%20%D1%86%D1%96%D0%BB%D1%8C%20_%20%D0%B7%D0%B2%D1%96%D1%82%29%20.pdf" class="ndc-acr-download-link is-original">Ukraine Second NDC</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2025-11/2%20Ukraine%20NDC2_adj_v2.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Ukraine Second NDC</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 42px;"></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/2%20%D0%9D%D0%92%D0%922%20%D0%BF%D1%80%D0%BE%D1%94%D0%BA%D1%82%20%28%D0%B7%D0%BC%D1%96%D0%BD%D0%B5%D0%BD%D0%B0%20%D1%86%D1%96%D0%BB%D1%8C%20_%20%D0%B7%D0%B2%D1%96%D1%82%29%20.pdf" class="ndc-acr-download-link is-original">Ukraine Second NDC</a><a href="https://unfccc.int/sites/default/files/2025-11/2%20Ukraine%20NDC2_adj_v2.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Ukraine Second NDC</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">11/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/2%20%D0%9D%D0%92%D0%922%20%D0%BF%D1%80%D0%BE%D1%94%D0%BA%D1%82%20%28%D0%B7%D0%BC%D1%96%D0%BD%D0%B5%D0%BD%D0%B0%20%D1%86%D1%96%D0%BB%D1%8C%20_%20%D0%B7%D0%B2%D1%96%D1%82%29%20.pdf" class="ndc-acr-download-link is-original">Ukraine Second NDC</a><a href="https://unfccc.int/sites/default/files/2025-11/2%20Ukraine%20NDC2_adj_v2.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Ukraine Second NDC</a></div>
</td>
</tr>
<tr class="submission-nid-653157 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bdi_flag.gif?h=57cf074e&amp;itok=M1gP3ooO" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Burundi
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-11/CDN3.0%20%20BURUNDI.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN3.0%20%20BURUNDI.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN3.0%20%20BURUNDI.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Burundi NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-653146 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/btn_flag.gif?h=57cf074e&amp;itok=3EUOB0qp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bhutan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Southern Asia" data-region-id="5908"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Third%20NDC%20%28Provisional%29_10%20November%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan's NDC 3.0 (Provisional)</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Third%20NDC%20%28Provisional%29_10%20November%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan's NDC 3.0 (Provisional)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Third%20NDC%20%28Provisional%29_10%20November%202025.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bhutan's NDC 3.0 (Provisional)</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-653022 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/blr_flag_0.gif?h=da9490b2&amp;itok=qg9XIK2J" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Belarus
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span><span class="region-data" data-region-name="Eastern Europe" data-region-id="5911"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/2025-11/Republic%20of%20Belarus%20NDC%20for%202026-2035.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Republic of Belarus NDC for 2026-2035</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 104px;"><span class="field--name-field-set-item-language is-original" style="height: 104px; display: table-cell; vertical-align: middle;">Russian</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Republic%20of%20Belarus%20NDC%20for%202026-2035.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Republic of Belarus NDC for 2026-2035</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">10/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Republic%20of%20Belarus%20NDC%20for%202026-2035.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Republic of Belarus NDC for 2026-2035</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-652920 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/tur_flag.gif?h=57cf074e&amp;itok=Bsjl49a2" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Türkiye
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/The%20Second%20NDC%20of%20T%C3%BCrkiye.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Türkiye Second NDC (NDC 3.0)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/The%20Second%20NDC%20of%20T%C3%BCrkiye.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Türkiye Second NDC (NDC 3.0)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_0">0</span><span class="alt_0 ndc_submission_version_0">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">09/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/The%20Second%20NDC%20of%20T%C3%BCrkiye.pdf" class="ndc-acr-download-link is-original" hreflang="en"> Türkiye Second NDC (NDC 3.0)</a></div>
</td>
</tr>
<tr class="submission-nid-652811 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bhs_flag.gif?h=da9490b2&amp;itok=T2Q_8_YY" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bahamas
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/The%20Bahamas%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahamas NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 0px;"></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/The%20Bahamas%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahamas NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">07/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/The%20Bahamas%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="en">Bahamas NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-main"></div>
</td>
</tr>
<tr class="submission-nid-652813 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/pry_flag.gif?h=da9490b2&amp;itok=xSwda_X3" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Paraguay
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="margin-bottom: 8px; height: 0px;"></div>
<div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Spanish</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0%20%28Anexo%20t%C3%A9cnico%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC Technical Annex</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"></div>
<div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-11/Paraguay%20NDC%203.0%20%28Anexo%20t%C3%A9cnico%29.pdf" class="ndc-acr-download-link is-original" hreflang="es">Paraguay NDC Technical Annex</a></div>
</td>
</tr>
<tr class="submission-nid-652263 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/per_flag.gif?h=57cf074e&amp;itok=qorWi1PB" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Peru
<span class="region-data" data-region-name="Americas" data-region-id="5901"></span><span class="region-data" data-region-name="Latin America and the Caribbean" data-region-id="5902"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original">Spanish</span><a href="https://unfccc.int/sites/default/files/2025-11/Documento%20NDC%203.0_UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru’s Updated Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 146px;"><span class="field--name-field-set-item-language is-original" style="height: 146px; display: table-cell; vertical-align: middle;">Spanish</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Documento%20NDC%203.0_UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru’s Updated Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Documento%20NDC%203.0_UNFCCC.pdf" class="ndc-acr-download-link is-original" hreflang="es">Peru’s Updated Nationally Determined Contribution (NDC 3.0)</a></div>
</td>
</tr>
<tr class="submission-nid-652279 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/uzb_flag.gif?h=da9490b2&amp;itok=Uk1GSfcz" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Uzbekistan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Central Asia" data-region-id="5905"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">Russian</span><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC%20rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan NDC 3.0</a><span class="field--name-field-set-item-language is-translation">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">Russian</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC%20rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC%20rus.pdf" class="ndc-acr-download-link is-original" hreflang="ru">Uzbekistan NDC 3.0</a><a href="https://unfccc.int/sites/default/files/2025-11/Uzbekistan%20Third%20NDC.pdf" class="ndc-acr-download-link is-translation" hreflang="en">Uzbekistan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-652301 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/gin_flag.gif?h=57cf074e&amp;itok=R4VpVcwp" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Guinea
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Western Africa" data-region-id="5900"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">French</span><a href="https://unfccc.int/sites/default/files/2025-11/CDN%203.0%20DE%20LA%20REPUBLIQUE%20DE%20GUINEE.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">French</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN%203.0%20DE%20LA%20REPUBLIQUE%20DE%20GUINEE.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/CDN%203.0%20DE%20LA%20REPUBLIQUE%20DE%20GUINEE.pdf" class="ndc-acr-download-link is-original" hreflang="fr">Guinea NDC 3.0</a></div>
<div class="attachmentset is-not-addendum is-not-other is-not-main"></div>
</td>
</tr>
<tr class="submission-nid-652810 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fji_flag.gif?h=da9490b2&amp;itok=x6GQCrGd" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Fiji
<span class="region-data" data-region-name="Oceania" data-region-id="5915"></span><span class="region-data" data-region-name="Melanesia" data-region-id="5917"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Fiji%20NDC3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 41px;"><span class="field--name-field-set-item-language is-original" style="height: 41px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Fiji%20NDC3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">06/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Fiji%20NDC3.0_Final.pdf" class="ndc-acr-download-link is-original" hreflang="en">Fiji NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-652941 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cyp_flag_0.gif?h=27ba19da&amp;itok=BvfXakal" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Cyprus
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652942 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/cze_flag_0.gif?h=bfb37f4a&amp;itok=KTP99jan" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Czechia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652940 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/hrv_flag.gif?h=b650b131&amp;itok=yw1I8PNh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Croatia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652945 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/fin_flag.gif?h=d1720097&amp;itok=1364Ux9S" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Finland
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652943 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/dnk_flag.gif?h=c3f562c8&amp;itok=-uSGdQls" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Denmark
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652944 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/est_flag.gif?h=d1720097&amp;itok=Ehy_GjOJ" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Estonia
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-651964 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/moz_flag.gif?h=d4b38aaf&amp;itok=BzqOJiNF" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Mozambique
<span class="region-data" data-region-name="Africa" data-region-id="5895"></span><span class="region-data" data-region-name="Eastern Africa" data-region-id="5897"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/Mozambique%20ProvNDC_ENG.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique's Provisional NDC 3.0</a></div>

</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 83px;"><span class="field--name-field-set-item-language is-original" style="height: 83px; display: table-cell; vertical-align: middle;">English</span></div>

</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Mozambique%20ProvNDC_ENG.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique's Provisional NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-11/Submission_Letter_Mozambique.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission Letter Mozambique</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"><span class="ndc_submission_version_3">3</span><span class="alt_0 ndc_submission_version_3">N/A</span> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/Mozambique%20ProvNDC_ENG.pdf" class="ndc-acr-download-link is-original" hreflang="en">Mozambique's Provisional NDC 3.0</a></div>
<div class="attachmentset is-addendum is-not-other is-not-main"><a href="https://unfccc.int/sites/default/files/2025-11/Submission_Letter_Mozambique.pdf" class="ndc-acr-download-link is-original" hreflang="en">Submission Letter Mozambique</a></div>
</td>
</tr>
<tr class="submission-nid-652939 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bgr_flag_0.gif?h=57cf074e&amp;itok=HsdbUL2X" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Bulgaria
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652938 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/bel_flag.gif?h=37efeadd&amp;itok=_0IpmWAW" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Belgium
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652937 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/aut_flag.gif?h=f64c9c91&amp;itok=3qcsMW-w" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Austria
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
</tr>
<tr class="submission-nid-652217 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/aze_flag_0.gif?h=da9490b2&amp;itok=orSCTF7_" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Azerbaijan
<span class="region-data" data-region-name="Asia" data-region-id="5904"></span><span class="region-data" data-region-name="Western Asia" data-region-id="5909"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Report_Azerbaijan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Azerbaijan NDC 3.0</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 62px;"><span class="field--name-field-set-item-language is-original" style="height: 62px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Report_Azerbaijan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Azerbaijan NDC 3.0</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/NDC%203.0%20Report_Azerbaijan.pdf" class="ndc-acr-download-link is-original" hreflang="en">Azerbaijan NDC 3.0</a></div>
</td>
</tr>
<tr class="submission-nid-652041 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/eu_flag.gif?h=57cf074e&amp;itok=-JZLp2wh" width="57" height="35" alt="" typeof="Image" class="img-responsive">
European Union (EU)
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 209px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en"> The nationally determined contribution of the European Union and its Member States (EU NDC)</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 209px;"><span class="field--name-field-set-item-language is-original" style="height: 209px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" class="views-field views-field-nothing-1"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en"> The nationally determined contribution of the European Union and its Member States (EU NDC)</a></div>
</td>
<td headers="view-field-version-number-table-column" class="views-field views-field-field-version-number"> </td>
<td headers="view-field-vd-status-table-column" class="views-field views-field-field-vd-status">Active </td>
<td headers="view-field-document-sb-table-column" class="views-field views-field-field-document-sb is-active">05/11/2025 </td>
<td headers="view-nothing-2-table-column" class="views-field views-field-nothing-2"><div class="attachmentset is-not-addendum is-not-other is-main"><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en"> The nationally determined contribution of the European Union and its Member States (EU NDC)</a></div>
</td>
</tr>
<tr class="submission-nid-652955 processed">
<td headers="view-title-table-column" class="views-field views-field-title"> <img loading="lazy" src="/sites/default/files/styles/flag_bigger/public/flags/lux_flag.gif?h=27ba19da&amp;itok=2TqxIE2q" width="57" height="35" alt="" typeof="Image" class="img-responsive">
Luxembourg
<span class="region-data" data-region-name="Europe" data-region-id="5910"></span> </td>
<td headers="view-field-vd-attachments-table-column" class="views-field views-field-field-vd-attachments"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original">English</span><a href="https://unfccc.int/sites/default/files/2025-11/DK-2025-11-05%20EU%20NDC.pdf" class="ndc-acr-download-link is-original" hreflang="en">The NDC of the European Union and its Member States</a></div>
</td>
<td headers="view-nothing-table-column" class="views-field views-field-nothing"><div class="attachmentset is-not-addendum is-not-other is-main" style="height: 125px;"><span class="field--name-field-set-item-language is-original" style="height: 125px; display: table-cell; vertical-align: middle;">English</span></div>
</td>
<td headers="view-nothing-1-table-column" cla
"""

if __name__ == "__main__":
    download_pdfs_from_html(html_source)

Directory created: ndc_downloads_robust
Found 352 unique PDF links.
Attempting to download: OFICIO 141.pdf...
  [SUCCESS] Saved to ndc_downloads_robust/OFICIO 141.pdf
Attempting to download: Mexico_NDC_UNFCCC_update2022_FINAL.pdf...
  [SUCCESS] Saved to ndc_downloads_robust/Mexico_NDC_UNFCCC_update2022_FINAL.pdf
Attempting to download: Honduras Segunda Actualización de su NDC (2021-2030)serna_rev_final.pdf...
  [SUCCESS] Saved to ndc_downloads_robust/Honduras Segunda Actualización de su NDC (2021-2030)serna_rev_final.pdf
Attempting to download: TH NDC 3.0.pdf...
  [SUCCESS] Saved to ndc_downloads_robust/TH NDC 3.0.pdf
Attempting to download: Primera NDC Ecuador.pdf...
  [SUCCESS] Saved to ndc_downloads_robust/Primera NDC Ecuador.pdf
Attempting to download: HON NDC 3.0 2026 Oficial.pdf...
  [SUCCESS] Saved to ndc_downloads_robust/HON NDC 3.0 2026 Oficial.pdf
Attempting to download: Segunda Contribución Determinada a Nivel Nacional_CDN2.pdf...
  [SUCCESS] Saved to ndc_downloads_robust/